<a href="https://colab.research.google.com/github/SanskrutiMandavkar20/Indian-Standards-AI/blob/main/IndianStandardAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Fri Aug 28 18:39:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 77 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (313 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!ollama --version

Start the Ollama server

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server start command executed")

Ollama server start command executed


Check whether Ollama is running

In [ ]:
!curl http://127.0.0.1:11434/api/tags

{"models":[]}

Download Qwen

In [ ]:
!ollama pull qwen2.5:3b

Test the model

In [ ]:
!ollama run qwen2.5:3b "Explain in simple language what an Indian Standard is."

An Indian Standard is like a set of rules or guidelines that help make sure
sure things in India are made and used safely and correctly. These rules co
cover everything from small items like screws and bolts to big things like 
bridges and airplanes. They make sure that when you buy a product, it will 
work well and not harm you or the environment. Indian Standards help keep e
everyone safe and healthy by setting high quality and safety levels for pro
products and services.



Install the RAG libraries

In [ ]:
!pip install -q sentence-transformers faiss-cpu pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.5 MB/s eta 0:00:00


Test our embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


In [ ]:
test_text = "Fire resistant electrical wire for industrial building"

embedding = embedding_model.encode(test_text)

print("Embedding shape:", embedding.shape)
print("First 10 values:", embedding[:10])

Embedding shape: (384,)
First 10 values: [-0.03117321  0.11823696 -0.05161348  0.00951813  0.03034337  0.08809528
  0.02176108  0.00754545 -0.02130037  0.00159944]


Create sample standards data

In [ ]:
standards = [
    {
        "standard_id": "DEMO-001",
        "title": "Electrical cables for industrial applications",
        "scope": (
            "Requirements for electrical cables used in "
            "industrial installations, including conductor, "
            "insulation and performance requirements."
        ),
        "keywords": [
            "electrical cable",
            "electrical wire",
            "industrial",
            "cable"
        ]
    },
    {
        "standard_id": "DEMO-002",
        "title": "Fire performance requirements for electrical cables",
        "scope": (
            "Requirements and test methods related to flame "
            "propagation and fire performance of electrical cables."
        ),
        "keywords": [
            "fire resistant",
            "flame retardant",
            "fire performance",
            "electrical cable"
        ]
    },
    {
        "standard_id": "DEMO-003",
        "title": "Electrical installation safety in buildings",
        "scope": (
            "Safety requirements for electrical installations "
            "in commercial and industrial buildings."
        ),
        "keywords": [
            "building",
            "industrial building",
            "electrical installation",
            "safety"
        ]
    },
    {
        "standard_id": "DEMO-004",
        "title": "LED street lighting luminaires",
        "scope": (
            "Performance requirements for LED luminaires used "
            "for road and street lighting."
        ),
        "keywords": [
            "LED",
            "street light",
            "luminaire",
            "road lighting"
        ]
    }
]

print(f"Loaded {len(standards)} sample standards")

Loaded 4 sample standards


Convert standards into searchable text

In [ ]:
documents = []

for standard in standards:

    text = f"""
Standard ID: {standard['standard_id']}
Title: {standard['title']}
Scope: {standard['scope']}
Keywords: {', '.join(standard['keywords'])}
"""

    documents.append(text.strip())

for document in documents:
    print(document)
    print("-" * 50)

Standard ID: DEMO-001
Title: Electrical cables for industrial applications
Scope: Requirements for electrical cables used in industrial installations, including conductor, insulation and performance requirements.
Keywords: electrical cable, electrical wire, industrial, cable
--------------------------------------------------
Standard ID: DEMO-002
Title: Fire performance requirements for electrical cables
Scope: Requirements and test methods related to flame propagation and fire performance of electrical cables.
Keywords: fire resistant, flame retardant, fire performance, electrical cable
--------------------------------------------------
Standard ID: DEMO-003
Title: Electrical installation safety in buildings
Scope: Safety requirements for electrical installations in commercial and industrial buildings.
Keywords: building, industrial building, electrical installation, safety
--------------------------------------------------
Standard ID: DEMO-004
Title: LED street lighting luminaires
S

Create embeddings and FAISS index

In [ ]:
import numpy as np
import faiss

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

# Normalize vectors for cosine similarity
faiss.normalize_L2(document_embeddings)

# Create FAISS index
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add document embeddings
index.add(document_embeddings)

print("FAISS index created successfully")
print("Number of indexed documents:", index.ntotal)

FAISS index created successfully
Number of indexed documents: 4


Search the FAISS database

In [ ]:
def search_standards(query, top_k=3):
    # Convert user query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search the FAISS index
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "standard": standards[idx],
            "document": documents[idx]
        })

    return results

Test with example

In [ ]:
user_query = (
    "I need fire-resistant electrical wire "
    "for an industrial building"
)

results = search_standards(
    user_query,
    top_k=3
)

print("USER QUERY:")
print(user_query)

print("\nRETRIEVED STANDARDS:")
print("=" * 60)

for i, result in enumerate(results, start=1):
    print(f"\n{i}. {result['standard']['standard_id']}")
    print("Title:", result['standard']['title'])
    print("Similarity score:", round(result['score'], 3))
    print("Scope:", result['standard']['scope'])

USER QUERY:
I need fire-resistant electrical wire for an industrial building

RETRIEVED STANDARDS:

1. DEMO-002
Title: Fire performance requirements for electrical cables
Similarity score: 0.537
Scope: Requirements and test methods related to flame propagation and fire performance of electrical cables.

2. DEMO-001
Title: Electrical cables for industrial applications
Similarity score: 0.304
Scope: Requirements for electrical cables used in industrial installations, including conductor, insulation and performance requirements.

3. DEMO-003
Title: Electrical installation safety in buildings
Similarity score: 0.266
Scope: Safety requirements for electrical installations in commercial and industrial buildings.


Convert retrieved results into context

In [ ]:
def create_rag_context(results):
    context = ""

    for i, result in enumerate(results, start=1):
        standard = result["standard"]

        context += f"""
STANDARD {i}

Standard ID: {standard['standard_id']}
Title: {standard['title']}
Scope: {standard['scope']}
Similarity Score: {result['score']:.3f}

"""
    return context.strip()

In [ ]:
rag_context = create_rag_context(results)

print(rag_context)

STANDARD 1

Standard ID: DEMO-002
Title: Fire performance requirements for electrical cables
Scope: Requirements and test methods related to flame propagation and fire performance of electrical cables.
Similarity Score: 0.537


STANDARD 2

Standard ID: DEMO-001
Title: Electrical cables for industrial applications
Scope: Requirements for electrical cables used in industrial installations, including conductor, insulation and performance requirements.
Similarity Score: 0.304


STANDARD 3

Standard ID: DEMO-003
Title: Electrical installation safety in buildings
Scope: Safety requirements for electrical installations in commercial and industrial buildings.
Similarity Score: 0.266


Create the RAG prompt

In [ ]:
def create_rag_prompt(user_query, rag_context):

    prompt = f"""
You are an AI assistant for recommending applicable Indian Standards
for government and organizational procurement.

A user has provided this procurement requirement:

"{user_query}"

Below are the standards retrieved from the knowledge base:

{rag_context}

Your task is to analyze the user's requirement using ONLY the
retrieved standards.

STRICT RULES:

1. Do not invent any standard ID.
2. Do not mention any standard that is not present in the retrieved context.
3. Do not claim that a standard covers a requirement unless the
   provided scope explicitly supports that claim.
4. Clearly distinguish between:
   - Primary Recommendation
   - Allied Recommendations
5. If the retrieved information is insufficient, say so clearly.
6. Identify important missing information that may be required
   before selecting a final procurement specification.
7. Do not assume that "fire-resistant", "flame-retardant",
   "low-smoke", and "fire-survival" mean the same thing.
8. Use simple, professional language.

Return the answer in this format:

REQUIREMENT ANALYSIS
- Product:
- Application:
- Main requirement:

PRIMARY RECOMMENDATION
- Standard ID:
- Title:
- Reason:

ALLIED RECOMMENDATIONS
- Standard ID:
- Title:
- Reason:

MISSING INFORMATION
- ...

EVIDENCE LIMITATION
- ...

IMPORTANT:
The retrieved DEMO IDs are demonstration data only.
Do not present them as actual Indian Standards.
"""

    return prompt

Send the RAG prompt to Qwen

In [ ]:
prompt = create_rag_prompt(
    user_query,
    rag_context
)

print("Sending retrieved context to Qwen...\n")

!ollama run qwen2.5:3b "$prompt"

Sending retrieved context to Qwen...

**REQUIREMENT ANALYSIS**
- Product: Fire-resistant electrical wire
- Application: Industrial building
- Main requirement: Fire-resistant electrical wire for use in an industrial
industrial building

**PRIMARY RECOMMENDATION**
- Standard ID: DEMO-002
- Title: Fire performance requirements for electrical cables
- Reason: The scope of this standard covers requirements and test methods r
related to flame propagation and fire performance of electrical cables, whi
which directly relates to the need for fire-resistant electrical wire. This
This is the most relevant standard to the user's requirement.

**ALLIED RECOMMENDATIONS**
- Standard ID: DEMO-001
- Title: Electrical cables for industrial applications
- Reason: Although this standard does not specifically address fire-resista
fire-resistant properties, it does cover requirements for electrical cables
cables used in industrial installations, which may indirectly support the n
need for fire-resistant el

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2
        }
    }
)

response_data = response.json()

print(response_data["response"])

REQUIREMENT ANALYSIS
- Product: Fire-resistant electrical wire
- Application: Industrial building
- Main requirement: The wire needs to be fire-resistant for use in an industrial building environment.

PRIMARY RECOMMENDATION
- Standard ID: DEMO-002
- Title: Fire performance requirements for electrical cables
- Reason: This standard covers requirements and test methods related to flame propagation and fire performance of electrical cables, which is directly relevant to the user's requirement for fire-resistant electrical wire.

ALLIED RECOMMENDATIONS
- Standard ID: DEMO-001
- Title: Electrical cables for industrial applications
- Reason: This standard provides requirements for electrical cables used in industrial installations, including conductor, insulation, and performance requirements. While it does not specifically address fire resistance, it is relevant to the user's requirement as it covers the broader context of industrial electrical cables.

MISSING INFORMATION
- Specific fire 

Create a proper standards database

In [ ]:
standards_database = [
    {
        "standard_id": "DEMO-001",
        "version": "2024",
        "status": "active",
        "title": "Electrical cables for industrial applications",
        "scope": (
            "Requirements for electrical cables used in industrial "
            "installations, including conductor, insulation and "
            "performance requirements."
        ),
        "keywords": [
            "electrical cable",
            "electrical wire",
            "industrial",
            "cable"
        ],
        "related_standards": [
            "DEMO-002",
            "DEMO-003"
        ],
        "supersedes": [],
        "amendments": [],
        "certification_required": False
    },
    {
        "standard_id": "DEMO-002",
        "version": "2024",
        "status": "active",
        "title": "Fire performance requirements for electrical cables",
        "scope": (
            "Requirements and test methods related to flame propagation "
            "and fire performance of electrical cables."
        ),
        "keywords": [
            "fire resistant",
            "flame retardant",
            "fire performance",
            "electrical cable"
        ],
        "related_standards": [
            "DEMO-001",
            "DEMO-003"
        ],
        "supersedes": [
            "DEMO-002:2010"
        ],
        "amendments": [],
        "certification_required": False
    },
    {
        "standard_id": "DEMO-003",
        "version": "2024",
        "status": "active",
        "title": "Electrical installation safety in buildings",
        "scope": (
            "Safety requirements for electrical installations "
            "in commercial and industrial buildings."
        ),
        "keywords": [
            "building",
            "industrial building",
            "electrical installation",
            "safety"
        ],
        "related_standards": [
            "DEMO-001",
            "DEMO-002"
        ],
        "supersedes": [],
        "amendments": [],
        "certification_required": False
    },
    {
        "standard_id": "DEMO-004",
        "version": "2024",
        "status": "active",
        "title": "LED street lighting luminaires",
        "scope": (
            "Performance requirements for LED luminaires used "
            "for road and street lighting."
        ),
        "keywords": [
            "LED",
            "street light",
            "luminaire",
            "road lighting"
        ],
        "related_standards": [],
        "supersedes": [],
        "amendments": [],
        "certification_required": False
    }
]

print(f"Standards database loaded: {len(standards_database)} records")

Standards database loaded: 4 records


Create a standard lookup function

In [ ]:
def normalize_standard_id(standard_id):
    """
    Normalize standard IDs for comparison.
    """

    if not standard_id:
        return ""

    return (
        standard_id
        .lower()
        .replace(" ", "")
        .replace("-", "")
    )


def find_standard(standard_id, database):

    normalized_query = normalize_standard_id(
        standard_id
    )

    for standard in database:

        normalized_id = normalize_standard_id(
            standard["standard_id"]
        )

        if normalized_query == normalized_id:
            return standard

    return None

In [ ]:
result = find_standard(
    "DEMO-002",
    standards_database
)

print(result)

{'standard_id': 'DEMO-002', 'version': '2024', 'status': 'active', 'title': 'Fire performance requirements for electrical cables', 'scope': 'Requirements and test methods related to flame propagation and fire performance of electrical cables.', 'keywords': ['fire resistant', 'flame retardant', 'fire performance', 'electrical cable'], 'related_standards': ['DEMO-001', 'DEMO-003'], 'supersedes': ['DEMO-002:2010'], 'amendments': [], 'certification_required': False}


Create the tender IS-reference extractor

In [ ]:
import re

def extract_standard_references(text):

    references = []

    # Demo format
    demo_pattern = (
        r'\bDEMO-\d+'
        r'(?:\s*:\s*\d{4})?'
    )

    matches = re.findall(
        demo_pattern,
        text,
        flags=re.IGNORECASE
    )

    for match in matches:

        references.append(
            match.upper().strip()
        )

    # Remove duplicates
    return list(dict.fromkeys(references))

In [ ]:
sample_tender = """
Supply fire-resistant electrical cable for an industrial building.

The cable shall comply with DEMO-001:2010.

Fire performance requirements shall comply with DEMO-002:2010.
"""

mentioned_standards = extract_standard_references(
    sample_tender
)

print(mentioned_standards)

['DEMO-001:2010', 'DEMO-002:2010']


Create the tender audit engine

In [ ]:
def split_standard_and_version(reference):

    if ":" in reference:

        standard_id, version = reference.split(
            ":",
            1
        )

        return (
            standard_id.strip(),
            version.strip()
        )

    return reference.strip(), None


def audit_mentioned_standards(
    mentioned_standards,
    database
):

    audit_results = []

    for reference in mentioned_standards:

        standard_id, tender_version = (
            split_standard_and_version(reference)
        )

        standard = find_standard(
            standard_id,
            database
        )

        # Standard not found
        if not standard:

            audit_results.append({
                "standard_id": standard_id,
                "tender_version": tender_version,
                "status": "NOT_FOUND",
                "latest_version": None,
                "title": None,
                "recommendation": (
                    "This standard was not found in the "
                    "knowledge base. Verify the reference."
                )
            })

            continue

        latest_version = standard.get("version")
        standard_status = standard.get("status")

        # Withdrawn standard
        if standard_status == "withdrawn":

            result_status = "WITHDRAWN"

            recommendation = (
                "This standard is marked as withdrawn. "
                "Use the replacement standard if available."
            )

        # Version comparison
        elif (
            tender_version
            and latest_version
            and tender_version.isdigit()
            and latest_version.isdigit()
            and int(tender_version) < int(latest_version)
        ):

            result_status = "OUTDATED"

            recommendation = (
                f"The tender references version "
                f"{tender_version}, while the knowledge base "
                f"contains version {latest_version}."
            )

        else:

            result_status = "CURRENT"

            recommendation = (
                "The referenced standard matches the current "
                "version stored in the knowledge base."
            )

        audit_results.append({
            "standard_id": standard_id,
            "tender_version": tender_version,
            "latest_version": latest_version,
            "status": result_status,
            "title": standard.get("title"),
            "recommendation": recommendation
        })

    return audit_results

Test the complete audit system

In [ ]:
audit_results = audit_mentioned_standards(
    mentioned_standards,
    standards_database
)

for result in audit_results:

    print("=" * 60)

    print("Standard:", result["standard_id"])
    print("Tender version:", result["tender_version"])
    print("Latest version:", result["latest_version"])
    print("Status:", result["status"])
    print("Title:", result["title"])
    print("Recommendation:")
    print(result["recommendation"])

Standard: DEMO-001
Tender version: 2010
Latest version: 2024
Status: OUTDATED
Title: Electrical cables for industrial applications
Recommendation:
The tender references version 2010, while the knowledge base contains version 2024.
Standard: DEMO-002
Tender version: 2010
Latest version: 2024
Status: OUTDATED
Title: Fire performance requirements for electrical cables
Recommendation:
The tender references version 2010, while the knowledge base contains version 2024.


Get recommendations from RAG for the tender

In [ ]:
def create_database_documents(database):
    documents = []

    for standard in database:

        text = f"""
Standard ID: {standard['standard_id']}
Version: {standard['version']}
Status: {standard['status']}
Title: {standard['title']}
Scope: {standard['scope']}
Keywords: {', '.join(standard['keywords'])}
"""

        documents.append(text.strip())

    return documents


database_documents = create_database_documents(
    standards_database
)

print(f"Created {len(database_documents)} searchable documents")

Created 4 searchable documents


Rebuild the FAISS index

In [ ]:
database_embeddings = embedding_model.encode(
    database_documents,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(database_embeddings)

dimension = database_embeddings.shape[1]

database_index = faiss.IndexFlatIP(dimension)

database_index.add(database_embeddings)

print("New FAISS index created")
print("Indexed standards:", database_index.ntotal)

New FAISS index created
Indexed standards: 4


Create a new search function

In [ ]:
def search_database_standards(
    query,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = database_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx == -1:
            continue

        results.append({
            "standard": standards_database[idx],
            "score": float(score)
        })

    return results

Search using the tender requirement

In [ ]:
tender_requirement = """
Fire-resistant electrical cable
for use in an industrial building.
"""

recommended_results = search_database_standards(
    tender_requirement,
    top_k=3
)

for result in recommended_results:

    standard = result["standard"]

    print("=" * 60)
    print(
        "Standard:",
        standard["standard_id"]
    )
    print(
        "Title:",
        standard["title"]
    )
    print(
        "Score:",
        round(result["score"], 3)
    )

Standard: DEMO-002
Title: Fire performance requirements for electrical cables
Score: 0.629
Standard: DEMO-001
Title: Electrical cables for industrial applications
Score: 0.412
Standard: DEMO-003
Title: Electrical installation safety in buildings
Score: 0.334


Compare tender vs RAG recommendations

In [ ]:
def compare_tender_with_recommendations(
    mentioned_standards,
    recommended_results
):

    # Extract IDs from tender
    tender_ids = set()

    for reference in mentioned_standards:

        standard_id, _ = (
            split_standard_and_version(reference)
        )

        tender_ids.add(
            normalize_standard_id(standard_id)
        )

    # Extract IDs from recommendations
    recommended_ids = set()

    for result in recommended_results:

        standard = result["standard"]

        recommended_ids.add(
            normalize_standard_id(
                standard["standard_id"]
            )
        )

    # Find missing standards
    missing_ids = (
        recommended_ids - tender_ids
    )

    missing_standards = []

    for result in recommended_results:

        standard = result["standard"]

        normalized_id = normalize_standard_id(
            standard["standard_id"]
        )

        if normalized_id in missing_ids:

            missing_standards.append({
                "standard_id": standard["standard_id"],
                "version": standard["version"],
                "title": standard["title"],
                "scope": standard["scope"],
                "similarity_score": result["score"]
            })

    return missing_standards

Test missing standards detection

In [ ]:
missing_standards = (
    compare_tender_with_recommendations(
        mentioned_standards,
        recommended_results
    )
)

print("\nPOTENTIALLY MISSING STANDARDS")
print("=" * 60)

if not missing_standards:

    print(
        "No additional standards were found "
        "by the current retrieval results."
    )

else:

    for standard in missing_standards:

        print(
            f"\nStandard: "
            f"{standard['standard_id']}"
        )

        print(
            f"Version: "
            f"{standard['version']}"
        )

        print(
            f"Title: "
            f"{standard['title']}"
        )

        print(
            f"Scope: "
            f"{standard['scope']}"
        )

        print(
            f"Similarity: "
            f"{standard['similarity_score']:.3f}"
        )


POTENTIALLY MISSING STANDARDS

Standard: DEMO-003
Version: 2024
Title: Electrical installation safety in buildings
Scope: Safety requirements for electrical installations in commercial and industrial buildings.
Similarity: 0.334


Create structured tender audit data

In [ ]:
import json

def prepare_audit_report_data(
    audit_results,
    missing_standards
):
    return {
        "mentioned_standards_audit": audit_results,
        "potentially_missing_standards": missing_standards
    }


audit_report_data = prepare_audit_report_data(
    audit_results,
    missing_standards
)

print(
    json.dumps(
        audit_report_data,
        indent=2
    )
)

{
  "mentioned_standards_audit": [
    {
      "standard_id": "DEMO-001",
      "tender_version": "2010",
      "latest_version": "2024",
      "status": "OUTDATED",
      "title": "Electrical cables for industrial applications",
      "recommendation": "The tender references version 2010, while the knowledge base contains version 2024."
    },
    {
      "standard_id": "DEMO-002",
      "tender_version": "2010",
      "latest_version": "2024",
      "status": "OUTDATED",
      "title": "Fire performance requirements for electrical cables",
      "recommendation": "The tender references version 2010, while the knowledge base contains version 2024."
    }
  ],
  "potentially_missing_standards": [
    {
      "standard_id": "DEMO-003",
      "version": "2024",
      "title": "Electrical installation safety in buildings",
      "scope": "Safety requirements for electrical installations in commercial and industrial buildings.",
      "similarity_score": 0.3336394429206848
    }
  ]
}


Create the final AI report prompt

In [ ]:
def create_tender_audit_prompt(
    tender_requirement,
    audit_report_data
):

    return f"""
You are an AI assistant for auditing procurement tender
specifications against an Indian Standards knowledge base.

PROCUREMENT REQUIREMENT:

{tender_requirement}

VERIFIED AUDIT RESULTS FROM THE SYSTEM:

{json.dumps(audit_report_data, indent=2)}

IMPORTANT RULES:

1. Treat the VERIFIED AUDIT RESULTS as the source of truth.
2. Do not invent any standard IDs.
3. Do not invent versions.
4. Do not say a standard is outdated unless its status in the
   verified audit results is OUTDATED.
5. Do not claim a potentially missing standard is mandatory.
   Use wording such as:
   "should be reviewed for applicability"
   or
   "may need to be considered."
6. The current dataset contains DEMO standards.
   Clearly label them as demonstration records and do not
   present them as real Indian Standards.
7. Use simple and professional language.

Generate the report in exactly this structure:

TENDER COMPLIANCE REPORT

PROCUREMENT REQUIREMENT
- Product/application:
- Key technical requirement:

STANDARDS ALREADY MENTIONED

For each standard:
- Standard ID
- Tender version
- Knowledge base version
- Status
- Recommendation

POTENTIALLY MISSING STANDARDS

For each:
- Standard ID
- Title
- Why it was retrieved
- Recommendation

OVERALL RECOMMENDATION

Include a short summary of the actions the procurement
official should take.

EVIDENCE LIMITATION

Clearly state that this result is based on the currently
available knowledge base and that the DEMO records are not
actual Indian Standards.
"""

Generate the final report with Qwen

In [ ]:
audit_prompt = create_tender_audit_prompt(
    tender_requirement,
    audit_report_data
)

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": audit_prompt,
        "stream": False,
        "options": {
            "temperature": 0.2
        }
    }
)

response.raise_for_status()

audit_ai_response = response.json()["response"]

print(audit_ai_response)

TENDER COMPLIANCE REPORT

PROCUREMENT REQUIREMENT
- Product/application: Fire-resistant electrical cable
- Key technical requirement: For use in an industrial building.

STANDARDS ALREADY MENTIONED

For each standard:
- Standard ID: DEMO-001
- Tender version: 2010
- Knowledge base version: 2024
- Status: OUTDATED
- Recommendation: The tender references version 2010, while the knowledge base contains version 2024.

- Standard ID: DEMO-002
- Tender version: 2010
- Knowledge base version: 2024
- Status: OUTDATED
- Recommendation: The tender references version 2010, while the knowledge base contains version 2024.

POTENTIALLY MISSING STANDARDS

For each:
- Standard ID: DEMO-003
- Title: Electrical installation safety in buildings
- Why it was retrieved: The standard ID is based on the knowledge base version 2024, which was retrieved for potential applicability.
- Recommendation: Should be reviewed for applicability.

OVERALL RECOMMENDATION

Based on the audit results, the tender references

Create the product recommendation prompt

In [ ]:
def create_recommendation_prompt(
    product_description,
    recommended_results
):

    retrieved_standards = []

    for result in recommended_results:

        standard = result["standard"]

        retrieved_standards.append({
            "standard_id": standard["standard_id"],
            "version": standard["version"],
            "status": standard["status"],
            "title": standard["title"],
            "scope": standard["scope"],
            "similarity_score": round(
                result["score"],
                3
            ),
            "related_standards": standard.get(
                "related_standards",
                []
            ),
            "certification_required": standard.get(
                "certification_required",
                False
            )
        })

    return f"""
You are an AI assistant that recommends applicable standards
for procurement specifications.

USER PROCUREMENT REQUIREMENT:

{product_description}

RETRIEVED KNOWLEDGE BASE RECORDS:

{json.dumps(retrieved_standards, indent=2)}

STRICT RULES:

1. Use ONLY the standards present in the retrieved knowledge base.
2. Do not invent standard IDs, versions, titles, or certification requirements.
3. Do not claim that a retrieved standard is mandatory unless the
   retrieved data explicitly says so.
4. Select a PRIMARY recommendation based on the closest match between
   the user's requirement and the retrieved standard scope.
5. Other relevant standards should be classified as ALLIED or RELATED.
6. If the information is insufficient, clearly mention what information
   is missing.
7. Do not confuse:
   - fire-resistant
   - flame-retardant
   - fire-survival
   - low-smoke
8. The current records are DEMO records. Do not present them as actual
   Indian Standards.

Return the answer using exactly this format:

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- ...

APPLICATION
- ...

KEY REQUIREMENTS
- ...

PRIMARY RECOMMENDED STANDARD
- Standard ID:
- Version:
- Title:
- Why it matches:

ALLIED STANDARDS TO REVIEW

For each standard:
- Standard ID:
- Title:
- Why it may be relevant:

MISSING INFORMATION

List information that should be clarified before finalizing
the procurement specification.

CERTIFICATION INFORMATION
- ...

EVIDENCE LIMITATION
- ...
"""

Test Mode 2

In [ ]:
product_description = """
We need to procure fire-resistant electrical cable
for use in an industrial building.
"""

In [ ]:
mode2_results = search_database_standards(
    product_description,
    top_k=3
)

for result in mode2_results:

    print(
        result["standard"]["standard_id"],
        "-",
        result["standard"]["title"],
        "| Score:",
        round(result["score"], 3)
    )

DEMO-002 - Fire performance requirements for electrical cables | Score: 0.636
DEMO-001 - Electrical cables for industrial applications | Score: 0.433
DEMO-003 - Electrical installation safety in buildings | Score: 0.343


Generate the recommendation

In [ ]:
recommendation_prompt = create_recommendation_prompt(
    product_description,
    mode2_results
)

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": recommendation_prompt,
        "stream": False,
        "options": {
            "temperature": 0.2
        }
    }
)

response.raise_for_status()

recommendation = response.json()["response"]

print(recommendation)

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Fire-resistant electrical cable

APPLICATION
- Use in an industrial building

KEY REQUIREMENTS
- Flame propagation and fire performance
- Compliance with industrial application standards

PRIMARY RECOMMENDED STANDARD
- Standard ID: DEMO-002
- Version: 2024
- Title: Fire performance requirements for electrical cables
- Why it matches: The scope of DEMO-002 covers requirements and test methods related to flame propagation and fire performance of electrical cables, which directly aligns with the user's requirement for fire-resistant electrical cable.

ALLIED STANDARDS TO REVIEW
- Standard ID: DEMO-001
- Title: Electrical cables for industrial applications
- Why it may be relevant: This standard covers requirements for electrical cables used in industrial installations, including conductor, insulation, and performance requirements, which can provide additional context for the user's procurement needs.

MISSING INFORMATION
- Clarification on speci

Create one unified recommendation function

In [ ]:
def recommend_standards(product_description, top_k=3):

    # Step 1: Search the standards database
    results = search_database_standards(
        product_description,
        top_k=top_k
    )

    # Step 2: Create AI prompt
    prompt = create_recommendation_prompt(
        product_description,
        results
    )

    # Step 3: Generate response using Qwen
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen2.5:3b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.2
            }
        }
    )

    response.raise_for_status()

    ai_response = response.json()["response"]

    return {
        "query": product_description,
        "retrieved_standards": results,
        "recommendation": ai_response
    }

In [ ]:
test_recommendation = recommend_standards(
    "We need waterproof LED street lights for municipal roads",
    top_k=3
)

print(test_recommendation["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- LED street lights for municipal roads

APPLICATION
- Municipal roads

KEY REQUIREMENTS
- Waterproof
- LED technology
- Suitable for street lighting
- Compliance with safety standards

PRIMARY RECOMMENDED STANDARD
- Standard ID: DEMO-004
- Version: 2024
- Title: LED street lighting luminaires
- Why it matches: This standard covers performance requirements for LED luminaires used for road and street lighting, which aligns with the user's requirement for LED street lights for municipal roads.

ALLIED STANDARDS TO REVIEW
- Standard ID: DEMO-001
- Title: Not specified
- Why it may be relevant: This standard may provide additional safety and performance requirements that are relevant to the user's requirement.

MISSING INFORMATION
- The current records do not specify the exact waterproofing requirements or the specific performance requirements for the LED street lights. This information is crucial for a comprehensive recommendation.

CERTIFICATION 

Create one unified tender audit function

In [ ]:
def audit_tender(
    tender_text,
    tender_requirement,
    top_k=3
):

    # Step 1: Extract standards mentioned in tender
    mentioned_standards = extract_standard_references(
        tender_text
    )

    # Step 2: Check their version and status
    audit_results = audit_mentioned_standards(
        mentioned_standards,
        standards_database
    )

    # Step 3: Search for relevant standards
    recommended_results = search_database_standards(
        tender_requirement,
        top_k=top_k
    )

    # Step 4: Find potentially missing standards
    missing_standards = (
        compare_tender_with_recommendations(
            mentioned_standards,
            recommended_results
        )
    )

    # Step 5: Prepare verified data
    audit_report_data = prepare_audit_report_data(
        audit_results,
        missing_standards
    )

    # Step 6: Create AI prompt
    prompt = create_tender_audit_prompt(
        tender_requirement,
        audit_report_data
    )

    # Step 7: Generate final report
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen2.5:3b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.2
            }
        }
    )

    response.raise_for_status()

    ai_report = response.json()["response"]

    return {
        "mentioned_standards": mentioned_standards,
        "audit_results": audit_results,
        "missing_standards": missing_standards,
        "report": ai_report
    }

In [ ]:
import requests
import subprocess
import time

# Function to check if Ollama server is running
def is_ollama_running():
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=1)
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        return False
    except requests.exceptions.Timeout:
        return False
    except Exception as e:
        print(f"An unexpected error occurred while checking Ollama: {e}")
        return False

# Ensure Ollama server is running
if not is_ollama_running():
    print("Ollama server not running. Attempting to start it...")
    # Attempt to restart Ollama server (ensure any previous process is killed if necessary, though `ollama serve` usually handles this)
    # This assumes 'ollama' is in PATH and the installation was successful.
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    time.sleep(10) # Give it more time to start up

    if is_ollama_running():
        print("Ollama server restarted successfully.")
    else:
        print("Failed to restart Ollama server. Please check your Ollama installation.")
        # If it still fails, raise an error or exit
        raise ConnectionError("Ollama server is not running and could not be restarted.")

sample_tender = """
TENDER FOR PROCUREMENT OF ELECTRICAL CABLES

The department requires fire-resistant electrical cables
for use in an industrial building.

The cables shall comply with DEMO-001:2010.

Fire performance shall comply with DEMO-002:2010.
"""

test_audit = audit_tender(
    tender_text=sample_tender,
    tender_requirement=(
        "Fire-resistant electrical cable "
        "for use in an industrial building"
    )
)

print(test_audit["report"])

TENDER COMPLIANCE REPORT

PROCUREMENT REQUIREMENT
- Product/application: Fire-resistant electrical cable for use in an industrial building
- Key technical requirement: The cable must meet fire performance requirements and electrical installation safety standards.

STANDARDS ALREADY MENTIONED

For each standard:
- Standard ID: DEMO-001
- Tender version: 2010
- Knowledge base version: 2024
- Status: OUTDATED
- Recommendation: The tender references version 2010, while the knowledge base contains version 2024. It is recommended to update the tender to reflect the latest version 2024.

For each standard:
- Standard ID: DEMO-002
- Tender version: 2010
- Knowledge base version: 2024
- Status: OUTDATED
- Recommendation: The tender references version 2010, while the knowledge base contains version 2024. It is recommended to update the tender to reflect the latest version 2024.

POTENTIALLY MISSING STANDARDS

For each:
- Standard ID: DEMO-003
- Title: Electrical installation safety in buildings


Save the FAISS index

In [ ]:
import os

os.makedirs("indian_standards_rag", exist_ok=True)

faiss.write_index(
    database_index,
    "indian_standards_rag/standards.index"
)

print("FAISS index saved successfully")

FAISS index saved successfully


Save the metadata database

In [ ]:
with open(
    "indian_standards_rag/standards_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        standards_database,
        file,
        indent=2,
        ensure_ascii=False
    )

print("Standards metadata saved successfully")

Standards metadata saved successfully



PHASE 2: REAL INDIAN STANDARDS KNOWLEDGE BASE
====================================================

In [ ]:
print("Embedding model:", embedding_model)
print("FAISS library loaded successfully")

Embedding model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
FAISS library loaded successfully


In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json
import os

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")
print("FAISS loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully
FAISS loaded successfully


In [ ]:
test_embedding = embedding_model.encode(
    "This is a test sentence"
)

print("Embedding shape:", test_embedding.shape)

Embedding shape: (384,)


In [ ]:
try:
    print("Demo database:", len(standards_database))
    print("Demo FAISS index:", database_index.ntotal)
except NameError:
    print("Demo system is not loaded in the current runtime.")

Demo database: 4
Demo FAISS index: 4


Create a separate REAL standards database

In [ ]:
# ============================================
# REAL INDIAN STANDARDS DATABASE
# ============================================

real_standards_database = [

    {
        "standard_id": "IS 10322",
        "part": "Part 5 / Section 3",
        "edition_year": 2012,

        "title": (
            "Luminaires Part 5 Particular Requirements "
            "Section 3 Luminaires for Road and Street Lighting"
        ),

        "scope": (
            "Particular requirements for luminaires used for "
            "road and street lighting."
        ),

        "domain": "electrical",

        "product_category": (
            "Road and street lighting luminaires"
        ),

        "keywords": [
            "road lighting",
            "street lighting",
            "street light",
            "LED street light",
            "luminaire",
            "outdoor lighting",
            "municipal lighting"
        ],

        "status": "unverified",
        "status_verified": False,

        "related_standards": [
            "IS 16107 (Part 2 / Section 2)"
        ],

        "normative_references": [],

        "supersedes": [],
        "superseded_by": None,

        "amendments": [],

        "certification": {
            "applicable": "unverified",
            "mandatory": "unverified",
            "scheme": None,
            "qco": None
        },

        "source": "Bureau of Indian Standards",

        "source_url": (
            "https://www.bis.gov.in/"
        ),

        "verification_status": "official_source_found",
        "verified_on": None
    },


    {
        "standard_id": "IS 16107",
        "part": "Part 2 / Section 2",
        "edition_year": 2017,

        "title": (
            "Luminaires Performance Part 2 Particular Requirements "
            "Section 2 LED Street Lighting Luminaire"
        ),

        "scope": (
            "Performance requirements for LED luminaires used "
            "for road and street lighting."
        ),

        "domain": "electrical",

        "product_category": (
            "LED street lighting luminaire"
        ),

        "keywords": [
            "LED street light",
            "LED street lighting",
            "road lighting",
            "street lighting",
            "LED luminaire",
            "municipal lighting",
            "outdoor LED light"
        ],

        "status": "unverified",
        "status_verified": False,

        "related_standards": [
            "IS 10322 (Part 5 / Section 3)"
        ],

        "normative_references": [],

        "supersedes": [],
        "superseded_by": None,

        "amendments": [],

        "certification": {
            "applicable": "unverified",
            "mandatory": "unverified",
            "scheme": None,
            "qco": None
        },

        "source": "Bureau of Indian Standards",

        "source_url": (
            "https://www.bis.gov.in/"
        ),

        "verification_status": "official_source_found",
        "verified_on": None
    },


    {
        "standard_id": "IS 7098",
        "part": "Part 1",
        "edition_year": 2025,

        "title": (
            "Cross-linked Polyethylene Insulated Thermoplastic "
            "Sheathed Cables — Specification Part 1"
        ),

        "scope": (
            "Specification for cross-linked polyethylene insulated "
            "thermoplastic sheathed electrical cables for "
            "low-voltage applications."
        ),

        "domain": "electrical",

        "product_category": (
            "XLPE insulated electrical cable"
        ),

        "keywords": [
            "electrical cable",
            "XLPE cable",
            "power cable",
            "low voltage cable",
            "industrial cable",
            "insulated cable",
            "electrical wire"
        ],

        "status": "unverified",
        "status_verified": False,

        "related_standards": [],

        "normative_references": [],

        "supersedes": [],
        "superseded_by": None,

        "amendments": [],

        "certification": {
            "applicable": "unverified",
            "mandatory": "unverified",
            "scheme": None,
            "qco": None
        },

        "source": "Bureau of Indian Standards",

        "source_url": (
            "https://www.bis.gov.in/"
        ),

        "verification_status": "official_source_found",
        "verified_on": None
    }
]

print(
    "Real standards loaded:",
    len(real_standards_database)
)

Real standards loaded: 3


Inspect the real database

In [ ]:
for standard in real_standards_database:

    print("=" * 60)
    print("Standard ID:", standard["standard_id"])
    print("Part:", standard["part"])
    print("Year:", standard["edition_year"])
    print("Title:", standard["title"])
    print("Domain:", standard["domain"])

Standard ID: IS 10322
Part: Part 5 / Section 3
Year: 2012
Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
Domain: electrical
Standard ID: IS 16107
Part: Part 2 / Section 2
Year: 2017
Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
Domain: electrical
Standard ID: IS 7098
Part: Part 1
Year: 2025
Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
Domain: electrical


Create RAG documents for the real standards

In [ ]:
def create_real_standard_document(standard):

    keywords = ", ".join(
        standard.get("keywords", [])
    )

    related = ", ".join(
        standard.get("related_standards", [])
    )

    document = f"""
Standard ID: {standard['standard_id']}

Part: {standard.get('part', '')}

Edition Year: {standard.get('edition_year', '')}

Title:
{standard.get('title', '')}

Scope:
{standard.get('scope', '')}

Domain:
{standard.get('domain', '')}

Product Category:
{standard.get('product_category', '')}

Keywords:
{keywords}

Related Standards:
{related}
"""

    return document.strip()

Create all real database documents

In [ ]:
real_database_documents = []

for standard in real_standards_database:

    document = create_real_standard_document(
        standard
    )

    real_database_documents.append(document)

In [ ]:
print(real_database_documents[0])

Standard ID: IS 10322

Part: Part 5 / Section 3

Edition Year: 2012

Title:
Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting

Scope:
Particular requirements for luminaires used for road and street lighting.

Domain:
electrical

Product Category:
Road and street lighting luminaires

Keywords:
road lighting, street lighting, street light, LED street light, luminaire, outdoor lighting, municipal lighting

Related Standards:
IS 16107 (Part 2 / Section 2)


Create embeddings for real standards

In [ ]:
real_database_embeddings = embedding_model.encode(
    real_database_documents,
    convert_to_numpy=True
).astype("float32")

In [ ]:
print(
    "Embedding shape:",
    real_database_embeddings.shape
)

Embedding shape: (3, 384)


Normalize the embeddings

In [ ]:
faiss.normalize_L2(
    real_database_embeddings
)

print("Embeddings normalized successfully")

Embeddings normalized successfully


Create a separate REAL FAISS index

In [ ]:
real_dimension = real_database_embeddings.shape[1]

real_database_index = faiss.IndexFlatIP(
    real_dimension
)

real_database_index.add(
    real_database_embeddings
)

print(
    "Real FAISS index created successfully"
)

print(
    "Number of indexed standards:",
    real_database_index.ntotal
)

Real FAISS index created successfully
Number of indexed standards: 3


Create the REAL standards search function

In [ ]:
def search_real_standards(query, top_k=3):

    # Prevent requesting more results
    # than available in the database
    top_k = min(
        top_k,
        len(real_standards_database)
    )

    # Create query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query
    faiss.normalize_L2(query_embedding)

    # Search FAISS
    scores, indices = real_database_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, index in zip(
        scores[0],
        indices[0]
    ):

        if index == -1:
            continue

        results.append({
            "score": float(score),
            "standard": real_standards_database[index]
        })

    return results

Test LED street-light retrieval

In [ ]:
query = """
We need waterproof and energy-efficient LED
street lights for municipal roads.
"""

results = search_real_standards(
    query,
    top_k=3
)

for result in results:

    standard = result["standard"]

    print("=" * 60)
    print("Standard:", standard["standard_id"])
    print("Part:", standard["part"])
    print("Year:", standard["edition_year"])
    print("Title:", standard["title"])
    print("Score:", round(result["score"], 3))

Standard: IS 10322
Part: Part 5 / Section 3
Year: 2012
Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
Score: 0.468
Standard: IS 16107
Part: Part 2 / Section 2
Year: 2017
Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
Score: 0.459
Standard: IS 7098
Part: Part 1
Year: 2025
Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
Score: 0.119


Test cable retrieval

In [ ]:
query = """
We need electrical cable for an industrial
low-voltage installation.
"""

results = search_real_standards(
    query,
    top_k=3
)

for result in results:

    standard = result["standard"]

    print("=" * 60)
    print("Standard:", standard["standard_id"])
    print("Title:", standard["title"])
    print("Score:", round(result["score"], 3))

Standard: IS 7098
Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
Score: 0.407
Standard: IS 10322
Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
Score: 0.126
Standard: IS 16107
Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
Score: 0.063


Create a helper function to display results

In [ ]:
def display_real_search_results(results):

    print("=" * 70)
    print("REAL INDIAN STANDARDS - RAG RESULTS")
    print("=" * 70)

    for i, result in enumerate(results, start=1):

        standard = result["standard"]

        print(f"\n{i}. {standard['standard_id']}")

        print(
            "Part:",
            standard.get("part")
        )

        print(
            "Year:",
            standard.get("edition_year")
        )

        print(
            "Title:",
            standard.get("title")
        )

        print(
            "Domain:",
            standard.get("domain")
        )

        print(
            "Score:",
            round(result["score"], 3)
        )

        print(
            "Verification:",
            standard.get(
                "verification_status"
            )
        )

In [ ]:
results = search_real_standards(
    "LED street lights for municipal roads",
    top_k=3
)

display_real_search_results(results)

REAL INDIAN STANDARDS - RAG RESULTS

1. IS 10322
Part: Part 5 / Section 3
Year: 2012
Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
Domain: electrical
Score: 0.547
Verification: official_source_found

2. IS 16107
Part: Part 2 / Section 2
Year: 2017
Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
Domain: electrical
Score: 0.531
Verification: official_source_found

3. IS 7098
Part: Part 1
Year: 2025
Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
Domain: electrical
Score: 0.073
Verification: official_source_found


Save the real metadata database

In [ ]:
import os
import json

os.makedirs(
    "real_indian_standards_rag",
    exist_ok=True
)

In [ ]:
with open(
    "real_indian_standards_rag/standards_metadata.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        real_standards_database,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    "Real standards metadata saved successfully"
)

Real standards metadata saved successfully


Save the REAL FAISS index

In [ ]:
faiss.write_index(
    real_database_index,
    "real_indian_standards_rag/standards.index"
)

print(
    "Real FAISS index saved successfully"
)

Real FAISS index saved successfully


Create a separate AI recommendation function for REAL records

In [ ]:
def create_real_recommendation_prompt(
    product_description,
    retrieved_results
):

    standards = []

    for result in retrieved_results:

        standard = result["standard"]

        standards.append({
            "standard_id": standard["standard_id"],
            "part": standard.get("part"),
            "edition_year": standard.get(
                "edition_year"
            ),
            "title": standard.get("title"),
            "scope": standard.get("scope"),
            "status": standard.get("status"),
            "status_verified": standard.get(
                "status_verified"
            ),
            "certification": standard.get(
                "certification"
            ),
            "score": round(
                result["score"],
                3
            )
        })

    prompt = f"""
You are an AI-powered Indian Standards recommendation assistant
for procurement specifications.

USER REQUIREMENT:

{product_description}

RETRIEVED STANDARD RECORDS:

{json.dumps(standards, indent=2)}

STRICT RULES:

1. Use ONLY the retrieved standard records.
2. Do not invent Indian Standard IDs.
3. Do not invent edition years.
4. Do not claim that a standard is CURRENT, OUTDATED,
   WITHDRAWN, or SUPERSEDED if status_verified is False.
5. Do not claim certification is mandatory if certification
   information is marked unverified.
6. Recommend standards based on the relationship between the
   procurement requirement and the scope of the standard.
7. Clearly separate:
   - Primary recommendation
   - Allied standards to review
8. If information is insufficient, ask for or list the missing
   technical details.
9. Do not say that retrieval alone proves a standard is mandatory.

Return this format:

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- ...

APPLICATION
- ...

KEY REQUIREMENTS
- ...

PRIMARY STANDARD TO REVIEW
- Standard ID:
- Part:
- Edition:
- Title:
- Why it appears relevant:

ALLIED STANDARDS TO REVIEW
- Standard ID:
- Why it may be relevant:

MISSING TECHNICAL INFORMATION
- ...

STATUS AND CERTIFICATION NOTE
- Clearly state whether this information has been verified.

RECOMMENDATION LIMITATION
- This recommendation is based on retrieved knowledge-base
  metadata and requires official verification where fields are
  marked unverified.
"""

    return prompt

Create the REAL recommendation engine

In [ ]:
def recommend_real_standards(
    product_description,
    top_k=3
):

    # Step 1: Search real standards
    results = search_real_standards(
        product_description,
        top_k=top_k
    )

    # Step 2: Create prompt
    prompt = create_real_recommendation_prompt(
        product_description,
        results
    )

    # Step 3: Ask local Qwen model
    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen2.5:3b",
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.2
            }
        }
    )

    response.raise_for_status()

    ai_response = response.json()["response"]

    return {
        "query": product_description,
        "retrieved_standards": results,
        "recommendation": ai_response
    }

Test the complete REAL RAG pipeline

In [ ]:
test = recommend_real_standards(
    """
    We need waterproof and energy-efficient LED
    street lights for municipal roads.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Waterproof and energy-efficient LED street lights
- For municipal roads

APPLICATION
- Municipal roads

KEY REQUIREMENTS
- Waterproofing
- Energy efficiency

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it appears relevant: This standard specifically addresses the performance requirements for LED street lighting luminaires, which aligns with the need for waterproof and energy-efficient LED street lights for municipal roads.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
- Why it may be relevant: This standard provides particular requirements for luminaires used for road and street lighting, which could offer additional insights into the desi

In [ ]:
import requests

print("Requests library imported successfully")

Requests library imported successfully


In [ ]:
test = recommend_real_standards(
    """
    We need waterproof and energy-efficient LED
    street lights for municipal roads.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Waterproof and energy-efficient LED street lights for municipal roads.

APPLICATION
- These LED street lights are designed for municipal roads, requiring specific standards to ensure they meet the necessary performance and safety criteria.

KEY REQUIREMENTS
- Waterproofing
- Energy efficiency
- Compliance with municipal road lighting standards

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it appears relevant: This standard specifically addresses the performance requirements for LED street lighting luminaires, which aligns with the key requirements of waterproofing and energy efficiency. It also falls within the scope of municipal road lighting applications.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular

In [ ]:
import os
import json
import requests
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("All required libraries imported successfully")

All required libraries imported successfully


In [ ]:
test = recommend_real_standards(
    """
    We need waterproof and energy-efficient LED
    street lights for municipal roads.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Waterproof and energy-efficient LED street lights for municipal roads.

APPLICATION
- These LED street lights are intended for use on municipal roads, requiring specific standards to ensure they meet the required performance and safety standards.

KEY REQUIREMENTS
- Waterproofing
- Energy efficiency
- Compliance with municipal road lighting standards

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it appears relevant: This standard specifically addresses the performance requirements for LED street lighting luminaires, including energy efficiency and waterproofing, which are critical for municipal road applications.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road a

In [ ]:
!ollama --version

ollama version is 0.33.1


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [ ]:
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"qwen2.5:3b","model":"qwen2.5:3b","modified_at":"2026-08-28T18:40:11.668052872Z","size":1929912432,"digest":"357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"3.1B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":2048},"capabilities":["completion","tools"]}]}

In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED      
qwen2.5:3b    357c53fb659c    1.9 GB    4 minutes ago    


In [ ]:
response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": "Say hello in one short sentence.",
        "stream": False
    },
    timeout=120
)

print(response.json()["response"])

Hello! How can I assist you today?


In [ ]:
print("Status code:", response.status_code)
print("Full response:")
print(response.text)

Status code: 200
Full response:
{"model":"qwen2.5:3b","created_at":"2026-08-28T18:44:20.414933888Z","response":"Hello! How can I assist you today?","done":true,"done_reason":"stop","context":[151644,8948,198,2610,525,1207,16948,11,3465,553,54364,14817,13,1446,525,264,10950,17847,13,151645,198,151644,872,198,45764,23811,304,825,2805,11652,13,151645,198,151644,77091,198,9707,0,2585,646,358,7789,498,3351,30],"total_duration":231947578,"load_duration":1034337,"prompt_eval_count":36,"prompt_eval_duration":40894000,"eval_count":10,"eval_duration":151425000}


In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
try:
    response_data = response.json()

    if response.status_code == 200 and "response" in response_data:
        print("Qwen response:")
        print(response_data["response"])
    else:
        print("Ollama returned an error:")
        print(response_data)

except Exception as e:
    print("Could not process Ollama response:")
    print(e)
    print(response.text)

Qwen response:
Hello! How can I assist you today?


In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen2.5:3b    357c53fb659c    1.9 GB    Less than a second ago    


In [ ]:
!ollama run qwen2.5:3b "Say hello in one short sentence."

Hello! How can I assist you today?



In [ ]:
test = recommend_real_standards(
    """
    We need waterproof and energy-efficient LED
    street lights for municipal roads.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Waterproof and energy-efficient LED street lights for municipal roads.

APPLICATION
- Municipal roads

KEY REQUIREMENTS
- Waterproofing
- Energy efficiency

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it appears relevant: This standard specifically addresses the performance requirements for LED street lighting luminaires, which aligns with the need for waterproof and energy-efficient LED street lights for municipal roads.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
- Why it may be relevant: Although this standard is not specifically about LED street lights, it covers particular requirements for luminaires used for road and street lightin

In [ ]:
print("RETRIEVED STANDARDS")
print("=" * 60)

for item in test["retrieved_standards"]:

    standard = item["standard"]

    print("Standard:", standard["standard_id"])
    print("Title:", standard["title"])
    print("Score:", round(item["score"], 3))
    print("-" * 60)

RETRIEVED STANDARDS
Standard: IS 10322
Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
Score: 0.468
------------------------------------------------------------
Standard: IS 16107
Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
Score: 0.459
------------------------------------------------------------
Standard: IS 7098
Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
Score: 0.119
------------------------------------------------------------


In [ ]:
test = recommend_real_standards(
    """
    We are constructing a neew industrial facility and need electrical cables for
    power distribution. The cables should have suitable insulation and reliable
    performance for industrial use.

    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Electrical cables for power distribution

APPLICATION
- Suitable for industrial use

KEY REQUIREMENTS
- Suitable insulation
- Reliable performance

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 7098
- Part: Part 1
- Edition: 2025
- Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables - Specification Part 1
- Why it appears relevant: This standard specifically addresses cross-linked polyethylene insulated thermoplastic sheathed electrical cables, which are suitable for industrial applications and meet the key requirements of suitable insulation and reliable performance.

ALLIED STANDARDS TO REVIEW
- Standard ID: None
- Why it may be relevant: There are no other standards in the retrieved records that are directly related to the specific requirements of industrial electrical cables.

MISSING TECHNICAL INFORMATION
- None

STATUS AND CERTIFICATION NOTE
- The status of this standard is marked as "unverified" and the certificatio

In [ ]:
test = recommend_real_standards(
    """
    A municipal corporation is procuring energy-efficient LED
    luminaires for installation along public roads and streets.
    The lighting fixtures will be installed outdoors and exposed
    to environmental conditions.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Energy-efficient LED luminaires for installation along public roads and streets.

APPLICATION
- Outdoor installation, exposed to environmental conditions.

KEY REQUIREMENTS
- Energy efficiency
- Durability
- Environmental resistance
- Compliance with local regulations and standards

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
- Why it appears relevant: This standard covers particular requirements for luminaires used for road and street lighting, which aligns with the application of energy-efficient LED luminaires for public road and street lighting.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it may be relevant: This standard focuses on 

In [ ]:
!nvidia-smi


Fri Aug 28 18:44:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             58W /   70W |    2535MiB /  15360MiB |     86%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q sentence-transformers faiss-cpu requests

In [ ]:
import os
import json
import time
import requests
import subprocess
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


In [ ]:
!ollama --version

ollama version is 0.33.1


In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED           
qwen2.5:3b    357c53fb659c    1.9 GB    About a minute ago    


In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen2.5:3b    357c53fb659c    1.9 GB    Less than a second ago    


In [ ]:
response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "qwen2.5:3b",
        "prompt": "Say hello in one short sentence.",
        "stream": False
    },
    timeout=120
)

print("Status:", response.status_code)
print(response.text)

Status: 200
{"model":"qwen2.5:3b","created_at":"2026-08-28T18:45:34.627200099Z","response":"Hello! How can I assist you today?","done":true,"done_reason":"stop","context":[151644,8948,198,2610,525,1207,16948,11,3465,553,54364,14817,13,1446,525,264,10950,17847,13,151645,198,151644,872,198,45764,23811,304,825,2805,11652,13,151645,198,151644,77091,198,9707,0,2585,646,358,7789,498,3351,30],"total_duration":216264435,"load_duration":1231916,"prompt_eval_count":36,"prompt_eval_duration":20121000,"eval_count":10,"eval_duration":153382000}


In [ ]:
test = recommend_real_standards(
    """
    We need waterproof and energy-efficient LED
    street lights for municipal roads.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Waterproof and energy-efficient LED street lights for municipal roads.

APPLICATION
- These LED street lights are intended for use on municipal roads, requiring specific standards to ensure they meet the necessary performance and safety requirements.

KEY REQUIREMENTS
- Waterproofing
- Energy efficiency
- Compliance with municipal lighting standards

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it appears relevant: This standard specifically addresses the performance requirements for LED street lighting luminaires, which aligns with the key requirements of waterproofing and energy efficiency for municipal roads.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road an

In [ ]:
test = recommend_real_standards(
    """
    We need fire-resistant electrical wires
    for a chemical manufacturing plant.
    The cables will be installed in areas
    exposed to high temperatures and fire risk.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Fire-resistant electrical wires
- For a chemical manufacturing plant

APPLICATION
- Installation in areas exposed to high temperatures and fire risk

KEY REQUIREMENTS
- Fire resistance
- High temperature resistance
- Compliance with electrical safety standards

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 7098
- Part: Part 1
- Edition: 2025
- Title: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables - Specification Part 1
- Why it appears relevant: This standard covers cross-linked polyethylene insulated thermoplastic sheathed cables, which are suitable for applications where cables are exposed to high temperatures and fire risk. The scope includes low-voltage applications, which aligns with the requirement for fire-resistant electrical wires in a chemical manufacturing plant.

ALLIED STANDARDS TO REVIEW
- Standard ID: None
- Why it may be relevant: There are no other standards in the retrieved records that directly address the

In [ ]:
test = recommend_real_standards(
    """
    We need cement for construction of a
    residential building.
    """,
    top_k=3
)

print(test["recommendation"])

PROCUREMENT REQUIREMENT ANALYSIS

PRODUCT
- Cement for construction of a residential building.

APPLICATION
- The product is used in the construction of a residential building, specifically for the foundation, walls, and other structural elements.

KEY REQUIREMENTS
- Durability
- Strength
- Workability
- Cost-effectiveness

PRIMARY STANDARD TO REVIEW
- Standard ID: IS 16107
- Part: Part 2 / Section 2
- Edition: 2017
- Title: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
- Why it appears relevant:
  - The title mentions "LED Street Lighting Luminaire," which is relevant to the application of cement in residential buildings, as cement is often used in the construction of residential lighting fixtures and infrastructure.

ALLIED STANDARDS TO REVIEW
- Standard ID: IS 10322
- Part: Part 5 / Section 3
- Edition: 2012
- Title: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
- Why it may be relevant:
  

In [ ]:
results = search_real_standards(
    """
    We need cement for construction of a
    residential building.
    """,
    top_k=5
)

for r in results:
    print(
        r["standard"]["standard_id"],
        "=>",
        round(r["score"], 3)
    )

IS 10322 => 0.133
IS 16107 => 0.092
IS 7098 => 0.002


Add a strict relevance filter

In [ ]:
def filter_relevant_standards(results, minimum_score=0.35):
    """
    Remove candidates whose semantic similarity is too low.
    """

    relevant = []

    for result in results:
        score = result["score"]

        if score >= minimum_score:
            relevant.append(result)

    return relevant

Create a safe search function

In [ ]:
def safe_search_real_standards(query, top_k=5, minimum_score=0.35):

    results = search_real_standards(
        query,
        top_k=top_k
    )

    relevant_results = filter_relevant_standards(
        results,
        minimum_score=minimum_score
    )

    return relevant_results

Test it with the cement query

In [ ]:
results = safe_search_real_standards(
    """
    We need cement for construction of a
    residential building.
    """,
    top_k=5
)

print("Relevant standards found:", len(results))

for result in results:

    standard = result["standard"]

    print(
        standard["standard_id"],
        "=>",
        round(result["score"], 3)
    )

Relevant standards found: 0


Test the LED query

In [ ]:
results = safe_search_real_standards(
    """
    We need waterproof and energy-efficient LED
    street lights for municipal roads.
    """,
    top_k=5
)

print("Relevant standards found:", len(results))

for result in results:

    standard = result["standard"]

    print(
        standard["standard_id"],
        "=>",
        round(result["score"], 3)
    )

Relevant standards found: 2
IS 10322 => 0.468
IS 16107 => 0.459


Test the fire-resistant cable query

In [ ]:
results = safe_search_real_standards(
    """
    We need fire-resistant electrical wires
    for a chemical manufacturing plant.
    """,
    top_k=5
)

print("Relevant standards found:", len(results))

for result in results:

    standard = result["standard"]

    print(
        standard["standard_id"],
        "=>",
        round(result["score"], 3)
    )

Relevant standards found: 0


modify the recommendation function

In [ ]:
def recommend_real_standards_safe(product_description, top_k=5):

    # --------------------------------------------------
    # STEP 1: Search the vector database
    # --------------------------------------------------

    results = search_real_standards(
        product_description,
        top_k=top_k
    )

    # --------------------------------------------------
    # STEP 2: Apply relevance threshold
    # --------------------------------------------------

    relevant_results = filter_relevant_standards(
        results,
        minimum_score=0.35
    )

    # --------------------------------------------------
    # STEP 3: No sufficiently relevant standard
    # --------------------------------------------------

    if len(relevant_results) == 0:

        return {
            "query": product_description,
            "retrieved_standards": results,
            "recommended_standards": [],
            "recommendation": """
PROCUREMENT REQUIREMENT ANALYSIS

No sufficiently relevant Indian Standard was found
in the current knowledge base.

The available standards did not meet the minimum
semantic relevance threshold for this requirement.

RECOMMENDATION

Do not make a definitive standard recommendation
from the current knowledge base.

The knowledge base should be expanded with standards
covering this product or application.
"""
        }

    # --------------------------------------------------
    # STEP 4: Only relevant results go to Qwen
    # --------------------------------------------------

    # Your existing Qwen recommendation code goes here.

    # For now, return the filtered candidates so
    # we can verify the retrieval first.

    return {
        "query": product_description,
        "retrieved_standards": results,
        "recommended_standards": relevant_results,
        "recommendation": "Relevant standards found. Ready for Qwen analysis."
    }

Test cement again

In [ ]:
test = recommend_real_standards_safe(
    """
    We need cement for construction of a
    residential building.
    """,
    top_k=5
)

print(test["recommendation"])


PROCUREMENT REQUIREMENT ANALYSIS

No sufficiently relevant Indian Standard was found
in the current knowledge base.

The available standards did not meet the minimum
semantic relevance threshold for this requirement.

RECOMMENDATION

Do not make a definitive standard recommendation
from the current knowledge base.

The knowledge base should be expanded with standards
covering this product or application.



Create requirement extraction

In [ ]:
def extract_requirements_for_rag(query):

    query_lower = query.lower()

    requirements = {
        "product_terms": [],
        "application_terms": [],
        "technical_requirements": []
    }

    # Product terms
    product_keywords = [
        "cable",
        "wire",
        "lighting",
        "luminaire",
        "switch",
        "transformer",
        "motor",
        "cement",
        "concrete"
    ]

    for keyword in product_keywords:
        if keyword in query_lower:
            requirements["product_terms"].append(keyword)

    # Application terms
    application_keywords = [
        "industrial",
        "residential",
        "commercial",
        "municipal",
        "road",
        "building",
        "factory",
        "outdoor",
        "indoor"
    ]

    for keyword in application_keywords:
        if keyword in query_lower:
            requirements["application_terms"].append(keyword)

    # Technical requirements
    technical_keywords = [
        "fire resistant",
        "fire resistance",
        "flame resistant",
        "high temperature",
        "waterproof",
        "water resistant",
        "energy efficient",
        "low voltage",
        "high voltage",
        "weather resistant"
    ]

    for keyword in technical_keywords:
        if keyword in query_lower:
            requirements["technical_requirements"].append(keyword)

    return requirements

Test it

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

requirements = extract_requirements_for_rag(query)

print(requirements)

{'product_terms': ['wire'], 'application_terms': [], 'technical_requirements': []}


Check whether a standard actually covers those requirements

In [ ]:
def check_requirement_coverage(query, standard):

    query_lower = query.lower()

    # Combine information available about the standard
    standard_text = " ".join([
        str(standard.get("standard_id", "")),
        str(standard.get("title", "")),
        str(standard.get("scope", "")),
        " ".join(standard.get("keywords", []))
    ]).lower()

    requirements = extract_requirements_for_rag(query)

    coverage = {}

    # Check each requirement
    for requirement in (
        requirements["product_terms"]
        + requirements["application_terms"]
        + requirements["technical_requirements"]
    ):

        if requirement in standard_text:
            coverage[requirement] = "SUPPORTED_BY_METADATA"
        else:
            coverage[requirement] = "NOT_CONFIRMED"

    return coverage

Test against IS 7098

In [ ]:
results = search_real_standards(
    """
    We need fire-resistant electrical wires
    for a chemical manufacturing plant.
    """,
    top_k=3
)

for result in results:

    standard = result["standard"]

    print("\nSTANDARD:", standard["standard_id"])
    print("TITLE:", standard["title"])
    print("SCORE:", round(result["score"], 3))

    coverage = check_requirement_coverage(
        """
        We need fire-resistant electrical wires
        for a chemical manufacturing plant.
        """,
        standard
    )

    print("COVERAGE:")

    for requirement, status in coverage.items():
        print(" ", requirement, ":", status)


STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
SCORE: 0.301
COVERAGE:
  wire : SUPPORTED_BY_METADATA

STANDARD: IS 10322
TITLE: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
SCORE: 0.073
COVERAGE:
  wire : NOT_CONFIRMED

STANDARD: IS 16107
TITLE: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
SCORE: 0.053
COVERAGE:
  wire : NOT_CONFIRMED


Replace the extractor with this improved version

In [ ]:
def extract_requirements_for_rag(query):

    query_lower = query.lower()

    requirements = {
        "product_terms": [],
        "application_terms": [],
        "technical_requirements": []
    }

    # --------------------------------------------------
    # PRODUCT TERMS
    # --------------------------------------------------

    product_keywords = [
        "wire",
        "wires",
        "cable",
        "cables",
        "electrical cable",
        "electrical wire",
        "lighting",
        "street light",
        "street lights",
        "led",
        "luminaire",
        "switch",
        "switchgear",
        "transformer",
        "motor",
        "cement",
        "concrete",
        "steel",
        "pipe",
        "pvc"
    ]

    for keyword in product_keywords:
        if keyword in query_lower:
            requirements["product_terms"].append(keyword)

    # --------------------------------------------------
    # APPLICATION / ENVIRONMENT
    # --------------------------------------------------

    application_keywords = [
        "industrial",
        "industrial building",
        "industrial plant",
        "manufacturing plant",
        "chemical plant",
        "chemical manufacturing plant",
        "factory",
        "residential",
        "residential building",
        "commercial",
        "commercial building",
        "municipal",
        "municipal road",
        "municipal roads",
        "road",
        "roads",
        "building",
        "outdoor",
        "indoor",
        "construction"
    ]

    for keyword in application_keywords:
        if keyword in query_lower:
            requirements["application_terms"].append(keyword)

    # --------------------------------------------------
    # TECHNICAL REQUIREMENTS
    # --------------------------------------------------

    technical_keywords = [
        "fire resistant",
        "fire-resistant",
        "fire resistance",
        "flame resistant",
        "flame-resistant",
        "flame resistance",
        "fire retardant",
        "fire-retardant",
        "flame retardant",
        "flame-retardant",
        "high temperature",
        "high-temperature",
        "temperature resistant",
        "temperature-resistant",
        "heat resistant",
        "heat-resistant",
        "waterproof",
        "water resistant",
        "water-resistant",
        "energy efficient",
        "energy-efficient",
        "low voltage",
        "low-voltage",
        "high voltage",
        "high-voltage",
        "weather resistant",
        "weather-resistant",
        "corrosion resistant",
        "corrosion-resistant"
    ]

    for keyword in technical_keywords:
        if keyword in query_lower:
            requirements["technical_requirements"].append(keyword)

    # Remove duplicates while preserving order
    for category in requirements:
        requirements[category] = list(
            dict.fromkeys(requirements[category])
        )

    return requirements

Test it again

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

requirements = extract_requirements_for_rag(query)

print(requirements)

{'product_terms': ['wire', 'wires', 'electrical wire'], 'application_terms': ['manufacturing plant', 'chemical manufacturing plant'], 'technical_requirements': ['fire-resistant']}


In [ ]:
query = """
We need waterproof and energy-efficient LED
street lights for municipal roads.
"""

requirements = extract_requirements_for_rag(query)

print(requirements)

{'product_terms': ['street light', 'street lights', 'led'], 'application_terms': ['municipal', 'municipal road', 'municipal roads', 'road', 'roads'], 'technical_requirements': ['waterproof', 'energy-efficient']}


In [ ]:
query = """
We need cement for construction of a
residential building.
"""

requirements = extract_requirements_for_rag(query)

print(requirements)

{'product_terms': ['cement'], 'application_terms': ['residential', 'residential building', 'building', 'construction'], 'technical_requirements': []}


Create semantic requirement matching

In [ ]:
import numpy as np

def semantic_similarity(text1, text2):
    """
    Calculate cosine similarity between two pieces of text
    using the existing embedding model.
    """

    emb1 = embedding_model.encode(
        text1,
        normalize_embeddings=True
    )

    emb2 = embedding_model.encode(
        text2,
        normalize_embeddings=True
    )

    return float(np.dot(emb1, emb2))

Test the embedding model

In [ ]:
score = semantic_similarity(
    "fire-resistant electrical cable",
    "resistance to flame propagation in electrical cables"
)

print("Semantic similarity:", round(score, 3))

Semantic similarity: 0.767


In [ ]:
score = semantic_similarity(
    "fire-resistant electrical cable",
    "LED street lighting luminaire"
)

print("Semantic similarity:", round(score, 3))

Semantic similarity: 0.065


Create requirement evidence text

In [ ]:
def get_standard_evidence_text(standard):

    parts = [
        str(standard.get("standard_id", "")),
        str(standard.get("title", "")),
        str(standard.get("scope", ""))
    ]

    keywords = standard.get("keywords", [])

    if keywords:
        parts.append(" ".join(keywords))

    return " ".join(parts)

Create semantic coverage analysis

In [ ]:
def semantic_requirement_coverage(query, standard):

    requirements = extract_requirements_for_rag(query)

    evidence_text = get_standard_evidence_text(standard)

    coverage = {}

    all_requirements = (
        requirements["product_terms"]
        + requirements["application_terms"]
        + requirements["technical_requirements"]
    )

    for requirement in all_requirements:

        score = semantic_similarity(
            requirement,
            evidence_text
        )

        coverage[requirement] = {
            "score": round(score, 3)
        }

    return coverage

Test it against IS 7098

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

results = search_real_standards(
    query,
    top_k=3
)

for result in results:

    standard = result["standard"]

    print("\n" + "=" * 60)
    print("STANDARD:", standard["standard_id"])
    print("TITLE:", standard["title"])
    print("FAISS SCORE:", round(result["score"], 3))

    coverage = semantic_requirement_coverage(
        query,
        standard
    )

    print("\nREQUIREMENT COVERAGE:")

    for requirement, data in coverage.items():

        print(
            requirement,
            "=>",
            data["score"]
        )


STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
FAISS SCORE: 0.301

REQUIREMENT COVERAGE:
wire => 0.27
wires => 0.278
electrical wire => 0.31
manufacturing plant => 0.031
chemical manufacturing plant => -0.015
fire-resistant => 0.23

STANDARD: IS 10322
TITLE: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
FAISS SCORE: 0.073

REQUIREMENT COVERAGE:
wire => 0.023
wires => 0.065
electrical wire => 0.03
manufacturing plant => -0.022
chemical manufacturing plant => -0.065
fire-resistant => -0.012

STANDARD: IS 16107
TITLE: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
FAISS SCORE: 0.053

REQUIREMENT COVERAGE:
wire => 0.001
wires => 0.043
electrical wire => -0.013
manufacturing plant => -0.02
chemical manufacturing plant => -0.06
fire-resistant => 0.02


In [ ]:
query = """
We need cement for construction of a
residential building.
"""

results = search_real_standards(
    query,
    top_k=3
)

for result in results:

    standard = result["standard"]

    print("\n" + "=" * 60)
    print("STANDARD:", standard["standard_id"])
    print("TITLE:", standard["title"])
    print("FAISS SCORE:", round(result["score"], 3))

    coverage = semantic_requirement_coverage(
        query,
        standard
    )

    print("\nREQUIREMENT COVERAGE:")

    for requirement, data in coverage.items():

        print(
            requirement,
            "=>",
            data["score"]
        )


STANDARD: IS 10322
TITLE: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
FAISS SCORE: 0.133

REQUIREMENT COVERAGE:
cement => -0.021
residential => 0.134
residential building => 0.152
building => 0.046
construction => 0.036

STANDARD: IS 16107
TITLE: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
FAISS SCORE: 0.092

REQUIREMENT COVERAGE:
cement => -0.036
residential => 0.132
residential building => 0.138
building => 0.062
construction => 0.062

STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
FAISS SCORE: 0.002

REQUIREMENT COVERAGE:
cement => -0.032
residential => 0.065
residential building => 0.092
building => 0.079
construction => 0.078


Create a coverage classifier

In [ ]:
def classify_coverage_score(score):

    if score >= 0.45:
        return "STRONG"

    elif score >= 0.25:
        return "MODERATE"

    elif score >= 0.10:
        return "WEAK"

    else:
        return "NOT_SUPPORTED"

Build a complete applicability report

In [ ]:
def analyze_standard_applicability(query, result):

    standard = result["standard"]

    coverage = semantic_requirement_coverage(
        query,
        standard
    )

    classified = {}

    for requirement, data in coverage.items():

        score = data["score"]

        classified[requirement] = {
            "score": score,
            "coverage": classify_coverage_score(score)
        }

    return {
        "standard_id": standard["standard_id"],
        "title": standard["title"],
        "faiss_score": round(result["score"], 3),
        "coverage": classified
    }

Test IS 7098

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

results = search_real_standards(
    query,
    top_k=3
)

for result in results:

    report = analyze_standard_applicability(
        query,
        result
    )

    print("\n" + "=" * 60)

    print(
        report["standard_id"],
        "-",
        report["title"]
    )

    print(
        "FAISS:",
        report["faiss_score"]
    )

    print("\nRequirement coverage:")

    for requirement, data in report["coverage"].items():

        print(
            f"{requirement:25} "
            f"{data['score']:>6} "
            f"{data['coverage']}"
        )


IS 7098 - Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
FAISS: 0.301

Requirement coverage:
wire                        0.27 MODERATE
wires                      0.278 MODERATE
electrical wire             0.31 MODERATE
manufacturing plant        0.031 NOT_SUPPORTED
chemical manufacturing plant -0.015 NOT_SUPPORTED
fire-resistant              0.23 WEAK

IS 10322 - Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
FAISS: 0.073

Requirement coverage:
wire                       0.023 NOT_SUPPORTED
wires                      0.065 NOT_SUPPORTED
electrical wire             0.03 NOT_SUPPORTED
manufacturing plant       -0.022 NOT_SUPPORTED
chemical manufacturing plant -0.065 NOT_SUPPORTED
fire-resistant            -0.012 NOT_SUPPORTED

IS 16107 - Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
FAISS: 0.053

Requirement coverage:
wire                       0.001 NOT

Now calculate an overall applicability score

In [ ]:
def calculate_applicability_score(report):

    coverage = report["coverage"]

    product_scores = []
    application_scores = []
    technical_scores = []

    query = ""

    # Determine category using requirement names
    for requirement, data in coverage.items():

        score = max(0, data["score"])

        if requirement in [
            "wire",
            "wires",
            "cable",
            "cables",
            "electrical wire",
            "electrical cable"
        ]:
            product_scores.append(score)

        elif requirement in [
            "industrial",
            "industrial building",
            "industrial plant",
            "manufacturing plant",
            "chemical plant",
            "chemical manufacturing plant",
            "factory",
            "residential",
            "residential building",
            "commercial",
            "commercial building",
            "municipal",
            "municipal road",
            "municipal roads",
            "building",
            "outdoor",
            "indoor",
            "construction"
        ]:
            application_scores.append(score)

        else:
            technical_scores.append(score)

    product_score = max(product_scores, default=0)
    application_score = max(application_scores, default=0)
    technical_score = max(technical_scores, default=0)

    overall = (
        0.45 * product_score +
        0.20 * application_score +
        0.35 * technical_score
    )

    return round(overall, 3)

Test it

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

results = search_real_standards(
    query,
    top_k=3
)

for result in results:

    report = analyze_standard_applicability(
        query,
        result
    )

    applicability = calculate_applicability_score(
        report
    )

    print(
        report["standard_id"],
        "FAISS:",
        report["faiss_score"],
        "Applicability:",
        applicability
    )

IS 7098 FAISS: 0.301 Applicability: 0.226
IS 10322 FAISS: 0.073 Applicability: 0.029
IS 16107 FAISS: 0.053 Applicability: 0.026


Create the decision layer

In [ ]:
def make_applicability_decision(score):

    if score >= 0.40:
        return "RECOMMEND"

    elif score >= 0.15:
        return "REVIEW / VERIFY"

    else:
        return "REJECT"

Test the three standards

In [ ]:
scores = {
    "IS 7098": 0.226,
    "IS 10322": 0.029,
    "IS 16107": 0.026
}

for standard_id, score in scores.items():

    decision = make_applicability_decision(score)

    print(
        standard_id,
        "| Applicability:",
        score,
        "| Decision:",
        decision
    )

IS 7098 | Applicability: 0.226 | Decision: REVIEW / VERIFY
IS 10322 | Applicability: 0.029 | Decision: REJECT
IS 16107 | Applicability: 0.026 | Decision: REJECT


Automate it instead of manually entering scores

In [ ]:
def evaluate_standard_candidates(query, top_k=5):

    results = search_real_standards(
        query,
        top_k=top_k
    )

    evaluated = []

    for result in results:

        # Keep candidates with even moderate similarity
        # Applicability scoring will decide whether they are useful.
        if result["score"] < 0.05:
            continue

        report = analyze_standard_applicability(
            query,
            result
        )

        applicability_score = calculate_applicability_score(
            report
        )

        decision = make_applicability_decision(
            applicability_score
        )

        evaluated.append({
            "standard_id": report["standard_id"],
            "title": report["title"],
            "faiss_score": report["faiss_score"],
            "applicability_score": applicability_score,
            "decision": decision,
            "coverage": report["coverage"]
        })

    return evaluated

Test the entire pipeline

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

evaluated = evaluate_standard_candidates(
    query,
    top_k=5
)

print("Number of candidates:", len(evaluated))

for item in evaluated:

    print("\n" + "=" * 60)
    print("STANDARD:", item["standard_id"])
    print("TITLE:", item["title"])
    print("FAISS SCORE:", item["faiss_score"])
    print("APPLICABILITY:", item["applicability_score"])
    print("DECISION:", item["decision"])

Number of candidates: 3

STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1
FAISS SCORE: 0.301
APPLICABILITY: 0.226
DECISION: REVIEW / VERIFY

STANDARD: IS 10322
TITLE: Luminaires Part 5 Particular Requirements Section 3 Luminaires for Road and Street Lighting
FAISS SCORE: 0.073
APPLICABILITY: 0.029
DECISION: REJECT

STANDARD: IS 16107
TITLE: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
FAISS SCORE: 0.053
APPLICABILITY: 0.026
DECISION: REJECT


In [ ]:
def evaluate_standard_candidates(query, top_k=5):

    results = search_real_standards(
        query,
        top_k=top_k
    )

    evaluated = []

    for result in results:

        # Very low similarity → reject immediately
        if result["score"] < 0.05:
            continue

        report = analyze_standard_applicability(
            query,
            result
        )

        applicability_score = calculate_applicability_score(
            report
        )

        decision = make_applicability_decision(
            applicability_score
        )

        evaluated.append({
            "standard_id": report["standard_id"],
            "title": report["title"],
            "faiss_score": report["faiss_score"],
            "applicability_score": applicability_score,
            "decision": decision,
            "coverage": report["coverage"]
        })

    return evaluated

In [ ]:
query = """
We need cement for construction of a
residential building.
"""

evaluated = evaluate_standard_candidates(
    query,
    top_k=5
)

print("Candidates:", len(evaluated))

for item in evaluated:

    print(
        item["standard_id"],
        "| FAISS:",
        item["faiss_score"],
        "| Applicability:",
        item["applicability_score"],
        "| Decision:",
        item["decision"]
    )

Candidates: 2
IS 10322 | FAISS: 0.133 | Applicability: 0.03 | Decision: REJECT
IS 16107 | FAISS: 0.092 | Applicability: 0.028 | Decision: REJECT


Create the new knowledge-base schema

In [ ]:
import pandas as pd
import json
import os
from datetime import datetime

KB_COLUMNS = [
    "standard_id",
    "part",
    "section",
    "edition",
    "title",
    "scope",
    "product_categories",
    "applications",
    "technical_requirements",
    "keywords",
    "related_standards",
    "amendments",
    "status",
    "status_verified",
    "certification_applicable",
    "mandatory",
    "qco",
    "source_url",
    "evidence",
    "last_verified"
]

print("Knowledge base schema created.")
print("Fields:", len(KB_COLUMNS))

Knowledge base schema created.
Fields: 20


Create the first real records

In [ ]:
real_standards = [

    {
        "standard_id": "IS 7098",
        "part": "Part 1",
        "section": "",
        "edition": "2025",
        "title": "Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts",

        "scope": """
Cross-linked polyethylene insulated thermoplastic sheathed cables
for working voltages up to and including 1100 volts.
""",

        "product_categories": [
            "electrical cables",
            "power cables",
            "XLPE cables",
            "electrical wire"
        ],

        "applications": [
            "electrical installations",
            "power distribution",
            "industrial electrical installations"
        ],

        "technical_requirements": [
            "conductor requirements",
            "insulation requirements",
            "sheath requirements",
            "electrical performance",
            "mechanical performance"
        ],

        "keywords": [
            "cable",
            "wire",
            "XLPE",
            "electrical cable",
            "power cable",
            "1100 V"
        ],

        "related_standards": [],
        "amendments": [],

        "status": "current",
        "status_verified": True,

        "certification_applicable": None,
        "mandatory": None,
        "qco": None,

        "source_url":
            "https://www.bis.gov.in/draft-test-request-and-test-report-formats/1st-team-reviewed-top-mand-iss-electrical/?lang=en",

        "evidence": """
BIS Electrical Discipline page lists IS 7098 (Part 1):2025
as Cross-linked Polyethylene Insulated Thermoplastic Sheathed
Cables — Specification Part 1 for working voltages up to and
including 1100 V (Second Revision).
BIS implementation guidance dated 27 August 2025 states that
IS 7098 Part 1:2025 revised the earlier 1988 edition.
""",

        "last_verified": "2026-08-28"
    },


    {
        "standard_id": "IS 10322",
        "part": "Part 5",
        "section": "Section 3",
        "edition": "2026",
        "title":
            "Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting",

        "scope": """
Requirements for luminaires used for road and street lighting
and other public outdoor lighting applications.
""",

        "product_categories": [
            "road lighting luminaires",
            "street lighting luminaires",
            "LED street lights",
            "outdoor lighting"
        ],

        "applications": [
            "road lighting",
            "street lighting",
            "public outdoor lighting",
            "municipal roads"
        ],

        "technical_requirements": [
            "luminaire performance",
            "electrical safety",
            "environmental protection",
            "lighting performance"
        ],

        "keywords": [
            "LED street light",
            "street lighting",
            "road lighting",
            "outdoor luminaire",
            "municipal lighting"
        ],

        "related_standards": [
            "IS 10322 Part 1"
        ],

        "amendments": [],

        "status": "current",
        "status_verified": True,

        "certification_applicable": None,
        "mandatory": None,
        "qco": None,

        "source_url":
            "https://lims.bis.gov.in/home/search_is_number/?is_number__doc_no=10322",

        "evidence": """
BIS LIMS currently lists IS 10322 (Part 5/Sec 3) (2026)
as Luminaires Part 5: Particular Requirements Section 3:
Luminaires for road and street lighting (Second Revision).
""",

        "last_verified": "2026-08-28"
    },


    {
        "standard_id": "IS 16107",
        "part": "Part 2",
        "section": "Section 2",
        "edition": "2017",
        "title":
            "Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire",

        "scope": """
Performance requirements for LED street lighting luminaires.
""",

        "product_categories": [
            "LED street lighting luminaires",
            "LED luminaires",
            "street lighting"
        ],

        "applications": [
            "street lighting",
            "road lighting",
            "outdoor lighting"
        ],

        "technical_requirements": [
            "luminaire performance",
            "LED performance",
            "electrical performance"
        ],

        "keywords": [
            "LED",
            "street light",
            "LED luminaire",
            "road lighting",
            "street lighting"
        ],

        "related_standards": [],

        "amendments": [],

        "status": "verified_record",
        "status_verified": True,

        "certification_applicable": None,
        "mandatory": None,
        "qco": None,

        "source_url":
            "https://lims.bis.gov.in/home/search_is_number/?is_number__doc_no=16107",

        "evidence": """
BIS LIMS lists IS 16107 Part 2 Section 2 (2017) for LED
street lighting luminaires.
""",

        "last_verified": "2026-08-28"
    },


    {
        "standard_id": "IS 269",
        "part": "",
        "section": "",
        "edition": "2015",
        "title":
            "Ordinary Portland Cement — Specification (Sixth Revision)",

        "scope": """
Specification requirements for Ordinary Portland Cement.
""",

        "product_categories": [
            "cement",
            "ordinary portland cement",
            "OPC"
        ],

        "applications": [
            "building construction",
            "residential construction",
            "civil construction",
            "concrete construction"
        ],

        "technical_requirements": [
            "fineness",
            "setting time",
            "soundness",
            "compressive strength",
            "chemical requirements"
        ],

        "keywords": [
            "cement",
            "OPC",
            "ordinary portland cement",
            "construction cement",
            "building cement"
        ],

        "related_standards": [
            "IS 4031",
            "IS 4032"
        ],

        "amendments": [],

        "status": "verified_record",
        "status_verified": True,

        "certification_applicable": None,
        "mandatory": None,
        "qco": None,

        "source_url":
            "https://www.bis.gov.in/is-269-2015/?lang=en",

        "evidence": """
BIS identifies IS 269:2015 as Ordinary Portland Cement —
Specification (Sixth Revision). BIS LIMS also lists testing
against IS 269:2015.
""",

        "last_verified": "2026-08-28"
    }
]

Convert it to a DataFrame

In [ ]:
kb_df = pd.DataFrame(real_standards)

print("Number of real standards:", len(kb_df))

display(
    kb_df[
        [
            "standard_id",
            "part",
            "section",
            "edition",
            "title",
            "status"
        ]
    ]
)

Number of real standards: 4


,standard_id,part,section,edition,title,status
0,IS 7098,Part 1,,2025,Cross-linked Polyethylene Insulated Thermoplas...,current
1,IS 10322,Part 5,Section 3,2026,Luminaires — Part 5: Particular Requirements S...,current
2,IS 16107,Part 2,Section 2,2017,Luminaires Performance Part 2 Particular Requi...,verified_record
3,IS 269,,,2015,Ordinary Portland Cement — Specification (Sixt...,verified_record


Save the knowledge base

In [ ]:
os.makedirs("knowledge_base", exist_ok=True)

kb_path = "knowledge_base/indian_standards.json"

with open(kb_path, "w", encoding="utf-8") as f:
    json.dump(
        real_standards,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", kb_path)

Saved: knowledge_base/indian_standards.json


In [ ]:
kb_df.to_csv(
    "knowledge_base/indian_standards.csv",
    index=False
)

print("CSV saved.")

CSV saved.


Create the text that will actually be embedded

In [ ]:
def build_embedding_text(record):

    return f"""
Indian Standard: {record['standard_id']}
Part: {record['part']}
Section: {record['section']}
Edition: {record['edition']}

Title:
{record['title']}

Scope:
{record['scope']}

Product Categories:
{', '.join(record['product_categories'])}

Applications:
{', '.join(record['applications'])}

Technical Requirements:
{', '.join(record['technical_requirements'])}

Keywords:
{', '.join(record['keywords'])}

Related Standards:
{', '.join(record['related_standards'])}

Status:
{record['status']}

Evidence:
{record['evidence']}
""".strip()

Generate embedding documents

In [ ]:
embedding_documents = []

for record in real_standards:

    text = build_embedding_text(record)

    embedding_documents.append({
        "standard_id": record["standard_id"],
        "text": text
    })

print(
    embedding_documents[0]["text"]
)

Indian Standard: IS 7098
Part: Part 1
Section: 
Edition: 2025

Title:
Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts

Scope:

Cross-linked polyethylene insulated thermoplastic sheathed cables
for working voltages up to and including 1100 volts.


Product Categories:
electrical cables, power cables, XLPE cables, electrical wire

Applications:
electrical installations, power distribution, industrial electrical installations

Technical Requirements:
conductor requirements, insulation requirements, sheath requirements, electrical performance, mechanical performance

Keywords:
cable, wire, XLPE, electrical cable, power cable, 1100 V

Related Standards:


Status:
current

Evidence:

BIS Electrical Discipline page lists IS 7098 (Part 1):2025
as Cross-linked Polyethylene Insulated Thermoplastic Sheathed
Cables — Specification Part 1 for working voltages up to and
including 1100 V (Second Revision).
BI

Generate embeddings

In [ ]:
texts = [
    item["text"]
    for item in embedding_documents
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (4, 384)


Build a NEW FAISS index

In [ ]:
import faiss
import numpy as np

embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

dimension = embedding_matrix.shape[1]

real_faiss_index = faiss.IndexFlatIP(
    dimension
)

real_faiss_index.add(
    embedding_matrix
)

print(
    "FAISS index created."
)

print(
    "Number of vectors:",
    real_faiss_index.ntotal
)

FAISS index created.
Number of vectors: 4


Save the FAISS index

In [ ]:
faiss.write_index(
    real_faiss_index,
    "knowledge_base/real_indian_standards.faiss"
)

print("FAISS index saved.")

FAISS index saved.


In [ ]:
with open(
    "knowledge_base/embedding_documents.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        embedding_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Embedding metadata saved.")

Embedding metadata saved.


Test the NEW database

In [ ]:
query = """
We need cement for construction
of a residential building.
"""

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = real_faiss_index.search(
    np.asarray(query_embedding, dtype="float32"),
    4
)

for score, idx in zip(scores[0], indices[0]):

    record = real_standards[idx]

    print("\n" + "=" * 60)
    print(
        record["standard_id"],
        "| Score:",
        round(float(score), 3)
    )
    print(record["title"])


IS 269 | Score: 0.483
Ordinary Portland Cement — Specification (Sixth Revision)

IS 10322 | Score: 0.222
Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting

IS 16107 | Score: 0.13
Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire

IS 7098 | Score: 0.088
Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts


In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = real_faiss_index.search(
    np.asarray(query_embedding, dtype="float32"),
    4
)

for score, idx in zip(scores[0], indices[0]):

    record = real_standards[idx]

    print("\n" + "=" * 60)
    print(
        record["standard_id"],
        "| Score:",
        round(float(score), 3)
    )
    print(record["title"])


IS 7098 | Score: 0.34
Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts

IS 10322 | Score: 0.172
Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting

IS 16107 | Score: 0.123
Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire

IS 269 | Score: 0.089
Ordinary Portland Cement — Specification (Sixth Revision)


Let's inspect what the RAG actually retrieves

In [ ]:
def inspect_retrieval(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = real_faiss_index.search(
        np.asarray(query_embedding, dtype="float32"),
        min(top_k, len(real_standards))
    )

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        record = real_standards[idx]

        print("\n" + "=" * 70)
        print("RANK:", rank)
        print("STANDARD:", record["standard_id"])
        print("TITLE:", record["title"])
        print("SCORE:", round(float(score), 3))

        print("\nPRODUCTS:")
        print(", ".join(record["product_categories"]))

        print("\nAPPLICATIONS:")
        print(", ".join(record["applications"]))

        print("\nTECHNICAL REQUIREMENTS:")
        print(", ".join(record["technical_requirements"]))

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

inspect_retrieval(query)


RANK: 1
STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts
SCORE: 0.34

PRODUCTS:
electrical cables, power cables, XLPE cables, electrical wire

APPLICATIONS:
electrical installations, power distribution, industrial electrical installations

TECHNICAL REQUIREMENTS:
conductor requirements, insulation requirements, sheath requirements, electrical performance, mechanical performance

RANK: 2
STANDARD: IS 10322
TITLE: Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting
SCORE: 0.172

PRODUCTS:
road lighting luminaires, street lighting luminaires, LED street lights, outdoor lighting

APPLICATIONS:
road lighting, street lighting, public outdoor lighting, municipal roads

TECHNICAL REQUIREMENTS:
luminaire performance, electrical safety, environmental protection, lighting performance

RANK: 3
STANDARD: IS 16107
TITLE: Luminaires Performa

Test another important case

In [ ]:
query = """
We need LED street lights for a highway.
"""

inspect_retrieval(query)


RANK: 1
STANDARD: IS 10322
TITLE: Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting
SCORE: 0.588

PRODUCTS:
road lighting luminaires, street lighting luminaires, LED street lights, outdoor lighting

APPLICATIONS:
road lighting, street lighting, public outdoor lighting, municipal roads

TECHNICAL REQUIREMENTS:
luminaire performance, electrical safety, environmental protection, lighting performance

RANK: 2
STANDARD: IS 16107
TITLE: Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
SCORE: 0.489

PRODUCTS:
LED street lighting luminaires, LED luminaires, street lighting

APPLICATIONS:
street lighting, road lighting, outdoor lighting

TECHNICAL REQUIREMENTS:
luminaire performance, LED performance, electrical performance

RANK: 3
STANDARD: IS 269
TITLE: Ordinary Portland Cement — Specification (Sixth Revision)
SCORE: 0.122

PRODUCTS:
cement, ordinary portland cement, OPC

APPLICATIONS:
building constru

Test a difficult query

In [ ]:
query = """
We need fire-resistant cables
for an industrial chemical plant
operating at high temperature.
"""

inspect_retrieval(query)


RANK: 1
STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts
SCORE: 0.451

PRODUCTS:
electrical cables, power cables, XLPE cables, electrical wire

APPLICATIONS:
electrical installations, power distribution, industrial electrical installations

TECHNICAL REQUIREMENTS:
conductor requirements, insulation requirements, sheath requirements, electrical performance, mechanical performance

RANK: 2
STANDARD: IS 10322
TITLE: Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting
SCORE: 0.149

PRODUCTS:
road lighting luminaires, street lighting luminaires, LED street lights, outdoor lighting

APPLICATIONS:
road lighting, street lighting, public outdoor lighting, municipal roads

TECHNICAL REQUIREMENTS:
luminaire performance, electrical safety, environmental protection, lighting performance

RANK: 3
STANDARD: IS 269
TITLE: Ordinary Portland Ce

Improve the IS 7098 record

In [ ]:
for record in real_standards:

    if record["standard_id"] == "IS 7098":

        record["product_categories"].extend([
            "FR cable",
            "FRLSH cable",
            "LSHF cable"
        ])

        record["technical_requirements"].extend([
            "fire performance",
            "flame resistance",
            "FR cable requirements",
            "FRLSH cable requirements",
            "LSHF cable requirements"
        ])

        record["keywords"].extend([
            "fire resistant cable",
            "fire resistance",
            "FR",
            "FRLSH",
            "LSHF",
            "high temperature cable"
        ])

print("IS 7098 enriched.")

IS 7098 enriched.


In [ ]:
for record in real_standards:

    if record["standard_id"] == "IS 7098":

        print(record["product_categories"])
        print(record["technical_requirements"])
        print(record["keywords"])

['electrical cables', 'power cables', 'XLPE cables', 'electrical wire', 'FR cable', 'FRLSH cable', 'LSHF cable']
['conductor requirements', 'insulation requirements', 'sheath requirements', 'electrical performance', 'mechanical performance', 'fire performance', 'flame resistance', 'FR cable requirements', 'FRLSH cable requirements', 'LSHF cable requirements']
['cable', 'wire', 'XLPE', 'electrical cable', 'power cable', '1100 V', 'fire resistant cable', 'fire resistance', 'FR', 'FRLSH', 'LSHF', 'high temperature cable']


We changed the data, so embeddings are now outdated

In [ ]:
embedding_documents = []

for record in real_standards:

    embedding_documents.append({
        "standard_id": record["standard_id"],
        "text": build_embedding_text(record)
    })

texts = [
    item["text"]
    for item in embedding_documents
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

real_faiss_index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

real_faiss_index.add(
    embedding_matrix
)

print(
    "New FAISS index:",
    real_faiss_index.ntotal,
    "vectors"
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

New FAISS index: 4 vectors


In [ ]:
faiss.write_index(
    real_faiss_index,
    "knowledge_base/real_indian_standards.faiss"
)

with open(
    "knowledge_base/embedding_documents.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        embedding_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Updated knowledge base saved.")

Updated knowledge base saved.


Retest fire-resistant cable

In [ ]:
query = """
We need fire-resistant electrical wires
for a chemical manufacturing plant.
"""

inspect_retrieval(query)


RANK: 1
STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts
SCORE: 0.384

PRODUCTS:
electrical cables, power cables, XLPE cables, electrical wire, FR cable, FRLSH cable, LSHF cable

APPLICATIONS:
electrical installations, power distribution, industrial electrical installations

TECHNICAL REQUIREMENTS:
conductor requirements, insulation requirements, sheath requirements, electrical performance, mechanical performance, fire performance, flame resistance, FR cable requirements, FRLSH cable requirements, LSHF cable requirements


IndexError: list index out of range

Create the domain structure

In [ ]:
DOMAINS = {
    "electrical": [
        "electrical cable",
        "electrical wire",
        "power cable",
        "electrical installation",
        "switchgear",
        "transformer",
        "LED lighting",
        "street lighting"
    ],

    "construction": [
        "cement",
        "concrete",
        "steel",
        "reinforcement",
        "brick",
        "building material",
        "aggregate"
    ],

    "fire_safety": [
        "fire extinguisher",
        "fire alarm",
        "fire resistant",
        "fire safety",
        "fire door",
        "fire protection"
    ],

    "ppe": [
        "safety helmet",
        "safety shoes",
        "protective clothing",
        "safety gloves",
        "eye protection",
        "respiratory protection"
    ],

    "plumbing": [
        "water pipe",
        "PVC pipe",
        "plumbing",
        "sanitary",
        "water supply"
    ],

    "mechanical": [
        "pump",
        "compressor",
        "valve",
        "pressure vessel",
        "industrial equipment"
    ]
}

print("Domains created:")
for domain, keywords in DOMAINS.items():
    print(domain, "→", len(keywords), "keywords")

Domains created:
electrical → 8 keywords
construction → 7 keywords
fire_safety → 6 keywords
ppe → 6 keywords
plumbing → 5 keywords
mechanical → 5 keywords


Create a master standard database

In [ ]:
MASTER_KB = []

for record in real_standards:

    new_record = record.copy()

    new_record["domain"] = "unknown"

    MASTER_KB.append(new_record)

print("Master KB records:", len(MASTER_KB))

Master KB records: 4


Automatically assign domains

In [ ]:
def assign_domain(record):

    text = " ".join([
        str(record.get("title", "")),
        str(record.get("scope", "")),
        " ".join(record.get("product_categories", [])),
        " ".join(record.get("keywords", []))
    ]).lower()

    domain_scores = {}

    for domain, keywords in DOMAINS.items():

        score = 0

        for keyword in keywords:

            if keyword.lower() in text:
                score += 1

        domain_scores[domain] = score

    best_domain = max(
        domain_scores,
        key=domain_scores.get
    )

    if domain_scores[best_domain] == 0:
        return "unknown"

    return best_domain

In [ ]:
for record in MASTER_KB:
    record["domain"] = assign_domain(record)

for record in MASTER_KB:

    print(
        record["standard_id"],
        "→",
        record["domain"]
    )

IS 7098 → electrical
IS 10322 → electrical
IS 16107 → electrical
IS 269 → construction


Create a standard normalization function

In [ ]:
import re

def normalize_standard_id(standard_id):

    if not standard_id:
        return ""

    standard_id = str(standard_id).upper().strip()

    standard_id = standard_id.replace("INDIAN STANDARD", "")
    standard_id = standard_id.replace("IS-", "IS ")
    standard_id = standard_id.replace("IS.", "IS ")

    standard_id = re.sub(
        r"\s+",
        " ",
        standard_id
    )

    return standard_id.strip()

In [ ]:
tests = [
    "IS-7098",
    "is 7098",
    "IS. 7098",
    " IS 7098 "
]

for x in tests:
    print(x, "→", normalize_standard_id(x))

IS-7098 → IS 7098
is 7098 → IS 7098
IS. 7098 → IS 7098
 IS 7098  → IS 7098


Normalize every record

In [ ]:
for record in MASTER_KB:

    record["standard_id"] = normalize_standard_id(
        record["standard_id"]
    )

print("Standard IDs normalized.")

Standard IDs normalized.


Prevent duplicate standards

In [ ]:
def remove_duplicate_standards(records):

    unique = {}

    for record in records:

        key = (
            record["standard_id"],
            record.get("part", ""),
            record.get("section", ""),
            record.get("edition", "")
        )

        if key not in unique:
            unique[key] = record

    return list(unique.values())

In [ ]:
MASTER_KB = remove_duplicate_standards(
    MASTER_KB
)

print(
    "Unique standards:",
    len(MASTER_KB)
)

Unique standards: 4


Create a version comparison function

In [ ]:
def compare_standard_versions(
    tender_standard_id,
    tender_year,
    records
):

    matches = []

    tender_standard_id = normalize_standard_id(
        tender_standard_id
    )

    for record in records:

        if normalize_standard_id(
            record["standard_id"]
        ) == tender_standard_id:

            matches.append(record)

    if not matches:

        return {
            "found": False,
            "message": "Standard not found in knowledge base."
        }

    editions = []

    for record in matches:

        try:
            year = int(record["edition"])
            editions.append((year, record))

        except:
            pass

    if not editions:

        return {
            "found": True,
            "status": "VERSION_UNKNOWN",
            "records": matches
        }

    latest_year, latest_record = max(
        editions,
        key=lambda x: x[0]
    )

    tender_year = int(tender_year)

    if tender_year < latest_year:

        status = "OUTDATED"

    elif tender_year == latest_year:

        status = "CURRENT"

    else:

        status = "CHECK"

    return {
        "found": True,
        "status": status,
        "tender_year": tender_year,
        "latest_year": latest_year,
        "latest_record": latest_record
    }

Test version checking

In [ ]:
result = compare_standard_versions(
    "IS 7098",
    2010,
    MASTER_KB
)

print(result)

{'found': True, 'status': 'OUTDATED', 'tender_year': 2010, 'latest_year': 2025, 'latest_record': {'standard_id': 'IS 7098', 'part': 'Part 1', 'section': '', 'edition': '2025', 'title': 'Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts', 'scope': '\nCross-linked polyethylene insulated thermoplastic sheathed cables\nfor working voltages up to and including 1100 volts.\n', 'product_categories': ['electrical cables', 'power cables', 'XLPE cables', 'electrical wire', 'FR cable', 'FRLSH cable', 'LSHF cable'], 'applications': ['electrical installations', 'power distribution', 'industrial electrical installations'], 'technical_requirements': ['conductor requirements', 'insulation requirements', 'sheath requirements', 'electrical performance', 'mechanical performance', 'fire performance', 'flame resistance', 'FR cable requirements', 'FRLSH cable requirements', 'LSHF cable requirements'], 'keywords': ['ca

Add verification levels

In [ ]:
VERIFICATION_LEVELS = {
    "A": "Official BIS source and evidence verified",
    "B": "Official BIS metadata verified",
    "C": "Secondary source; requires verification",
    "D": "Unverified"
}

In [ ]:
for record in MASTER_KB:

    if record.get("status_verified") is True:
        record["verification_level"] = "A"
    else:
        record["verification_level"] = "D"

Add evidence type

In [ ]:
for record in MASTER_KB:

    record["evidence_type"] = "official_bis"

Save the improved database

In [ ]:
os.makedirs(
    "knowledge_base",
    exist_ok=True
)

with open(
    "knowledge_base/master_indian_standards.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MASTER_KB,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Master knowledge base saved."
)

Master knowledge base saved.


Create a human-readable CSV

In [ ]:
rows = []

for record in MASTER_KB:

    rows.append({
        "standard_id": record.get("standard_id"),
        "part": record.get("part"),
        "section": record.get("section"),
        "edition": record.get("edition"),
        "title": record.get("title"),
        "domain": record.get("domain"),
        "status": record.get("status"),
        "verification_level":
            record.get("verification_level"),
        "source_url":
            record.get("source_url")
    })

master_df = pd.DataFrame(rows)

master_df.to_csv(
    "knowledge_base/master_indian_standards.csv",
    index=False
)

display(master_df)

,standard_id,part,section,edition,title,domain,status,verification_level,source_url
0,IS 7098,Part 1,,2025,Cross-linked Polyethylene Insulated Thermoplas...,electrical,current,A,https://www.bis.gov.in/draft-test-request-and-...
1,IS 10322,Part 5,Section 3,2026,Luminaires — Part 5: Particular Requirements S...,electrical,current,A,https://lims.bis.gov.in/home/search_is_number/...
2,IS 16107,Part 2,Section 2,2017,Luminaires Performance Part 2 Particular Requi...,electrical,verified_record,A,https://lims.bis.gov.in/home/search_is_number/...
3,IS 269,,,2015,Ordinary Portland Cement — Specification (Sixt...,construction,verified_record,A,https://www.bis.gov.in/is-269-2015/?lang=en


Create the BIS source collector

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import time
import pandas as pd

BIS_SEARCH_URL = "https://standards.bis.gov.in/website/standard-details"

print("BIS collector initialized.")

BIS collector initialized.


Create our candidate standard list

In [ ]:
BIS_CANDIDATES = {

    "electrical": [
        "IS 7098",
        "IS 8130",
        "IS 694",
        "IS 1554",
        "IS 1646",
        "IS 10322",
        "IS 16107"
    ],

    "construction": [
        "IS 269",
        "IS 455",
        "IS 1489",
        "IS 1489 Part 1",
        "IS 1489 Part 2",
        "IS 8041",
        "IS 8042",
        "IS 12330",
        "IS 12600",
        "IS 16415",
        "IS 16993",
        "IS 18189"
    ],

    "fire_safety": [
        "IS 15683",
        "IS 16018",
        "IS 2190",
        "IS 13039",
        "IS 1641",
        "IS 1642"
    ],

    "plumbing": [
        "IS 2556",
        "IS 774",
        "IS 7231",
        "IS 8931",
        "IS 12234",
        "IS 17585",
        "IS 8329"
    ],

    "ppe": [
        "IS 2925",
        "IS 15298"
    ]
}

total_candidates = sum(
    len(v) for v in BIS_CANDIDATES.values()
)

print("Total candidate standards:", total_candidates)

Total candidate standards: 34


Normalize candidate IDs

In [ ]:
def clean_is_number(value):

    value = str(value).upper().strip()

    value = value.replace("-", " ")
    value = re.sub(r"\s+", " ", value)

    return value

In [ ]:
for domain in BIS_CANDIDATES:

    BIS_CANDIDATES[domain] = [
        clean_is_number(x)
        for x in BIS_CANDIDATES[domain]
    ]

print(BIS_CANDIDATES)

{'electrical': ['IS 7098', 'IS 8130', 'IS 694', 'IS 1554', 'IS 1646', 'IS 10322', 'IS 16107'], 'construction': ['IS 269', 'IS 455', 'IS 1489', 'IS 1489 PART 1', 'IS 1489 PART 2', 'IS 8041', 'IS 8042', 'IS 12330', 'IS 12600', 'IS 16415', 'IS 16993', 'IS 18189'], 'fire_safety': ['IS 15683', 'IS 16018', 'IS 2190', 'IS 13039', 'IS 1641', 'IS 1642'], 'plumbing': ['IS 2556', 'IS 774', 'IS 7231', 'IS 8931', 'IS 12234', 'IS 17585', 'IS 8329'], 'ppe': ['IS 2925', 'IS 15298']}


Create a BIS record template

In [ ]:
def empty_bis_record():

    return {
        "standard_id": "",
        "part": "",
        "section": "",
        "edition": "",
        "title": "",
        "scope": "",
        "product_categories": [],
        "applications": [],
        "technical_requirements": [],
        "keywords": [],
        "related_standards": [],
        "amendments": [],
        "status": "",
        "status_verified": False,
        "certification_applicable": None,
        "mandatory": None,
        "qco": None,
        "source_url": "",
        "evidence": "",
        "last_verified": "",
        "domain": "",
        "verification_level": "D",
        "evidence_type": ""
    }

Create the first official source registry

In [ ]:
BIS_SOURCE_REGISTRY = {

    "cement": {
        "url":
        "https://www.bis.gov.in/wp-content/uploads/2025/05/COMPENDIUM-OF-CEMENT-STANDARDS.pdf",

        "type":
        "official_bis_compendium"
    },

    "plumbing": {
        "url":
        "https://www.bis.gov.in/wp-content/uploads/2022/01/ARTICLE-FOR-WHATS-NEW-NEW-STANDARDS-FOR-WATER-EFFICIENT-PLUMBING-PRODUCTS-NEW.pdf",

        "type":
        "official_bis_publication"
    },

    "fire_safety": {
        "url":
        "https://services.bis.gov.in/tmp/compendium_2025-06-02-03-56-50.pdf",

        "type":
        "official_bis_compendium"
    },

    "standards_portal": {
        "url":
        "https://standards.bis.gov.in/",

        "type":
        "official_bis_standards_portal"
    },

    "know_your_standard": {
        "url":
        "https://www.bis.gov.in/know-your-standard/?lang=en",

        "type":
        "official_bis_knowledge_portal"
    }
}

print("Official BIS sources registered.")

Official BIS sources registered.


Add source metadata to our current KB

In [ ]:
for record in MASTER_KB:

    record["source_authority"] = "Bureau of Indian Standards"

    if record.get("source_url"):
        record["evidence_type"] = "official_bis"

    record["last_verified"] = "2026-08-29"

Add the new real records

In [ ]:
new_records = [

    {
        "standard_id": "IS 455",
        "part": "",
        "section": "",
        "edition": "2015",
        "title": "Portland Slag Cement — Specification",
        "scope": "Specification requirements for Portland slag cement.",
        "product_categories": [
            "cement",
            "portland slag cement"
        ],
        "applications": [
            "construction",
            "building construction",
            "concrete construction"
        ],
        "technical_requirements": [
            "chemical requirements",
            "physical requirements",
            "strength",
            "setting time"
        ],
        "keywords": [
            "cement",
            "PSC",
            "portland slag cement",
            "construction cement"
        ],
        "related_standards": [],
        "amendments": [],
        "status": "verified_record",
        "status_verified": True,
        "certification_applicable": True,
        "mandatory": True,
        "qco": "Cement Quality Control Order",
        "source_url":
            "https://www.bis.gov.in/wp-content/uploads/2025/05/COMPENDIUM-OF-CEMENT-STANDARDS.pdf",
        "evidence":
            "Listed in the official BIS Compendium of Cement Standards.",
        "last_verified": "2026-08-29",
        "domain": "construction",
        "verification_level": "A",
        "evidence_type": "official_bis_compendium"
    },

    {
        "standard_id": "IS 1489",
        "part": "Part 1",
        "section": "",
        "edition": "2015",
        "title":
            "Portland Pozzolana Cement — Specification: Part 1 Fly Ash Based",
        "scope":
            "Specification requirements for fly ash based Portland pozzolana cement.",
        "product_categories": [
            "cement",
            "portland pozzolana cement",
            "PPC"
        ],
        "applications": [
            "construction",
            "building construction",
            "concrete construction"
        ],
        "technical_requirements": [
            "chemical requirements",
            "physical requirements",
            "strength",
            "setting time"
        ],
        "keywords": [
            "cement",
            "PPC",
            "fly ash cement",
            "portland pozzolana cement"
        ],
        "related_standards": [],
        "amendments": [],
        "status": "verified_record",
        "status_verified": True,
        "certification_applicable": True,
        "mandatory": True,
        "qco": "Cement Quality Control Order",
        "source_url":
            "https://www.bis.gov.in/wp-content/uploads/2025/05/COMPENDIUM-OF-CEMENT-STANDARDS.pdf",
        "evidence":
            "Listed in the official BIS Compendium of Cement Standards.",
        "last_verified": "2026-08-29",
        "domain": "construction",
        "verification_level": "A",
        "evidence_type": "official_bis_compendium"
    }
]

print("New verified records prepared:", len(new_records))

New verified records prepared: 2


Add fire-extinguisher records

In [ ]:
new_records.extend([

    {
        "standard_id": "IS 15683",
        "part": "",
        "section": "",
        "edition": "2018",
        "title":
            "Portable Fire Extinguishers",
        "scope":
            "Requirements applicable to portable fire extinguishers.",
        "product_categories": [
            "fire extinguisher",
            "portable fire extinguisher"
        ],
        "applications": [
            "fire safety",
            "industrial facilities",
            "commercial buildings",
            "building fire protection"
        ],
        "technical_requirements": [
            "fire extinguishing performance",
            "construction",
            "performance requirements"
        ],
        "keywords": [
            "fire extinguisher",
            "portable extinguisher",
            "fire protection"
        ],
        "related_standards": [],
        "amendments": [],
        "status": "verified_record",
        "status_verified": True,
        "certification_applicable": True,
        "mandatory": True,
        "qco": "Fire Extinguishers Quality Control Order, 2023",
        "source_url":
            "https://www.bis.gov.in/product-certification/products-under-compulsory-certification/scheme-i-mark-scheme/?lang=en",
        "evidence":
            "BIS Scheme-I information lists IS 15683:2018 for Portable Fire Extinguishers under the Fire Extinguishers Quality Control Order, 2023.",
        "last_verified": "2026-08-29",
        "domain": "fire_safety",
        "verification_level": "A",
        "evidence_type": "official_bis_certification"
    },

    {
        "standard_id": "IS 16018",
        "part": "",
        "section": "",
        "edition": "2012",
        "title":
            "Wheeled Fire Extinguishers",
        "scope":
            "Requirements applicable to wheeled fire extinguishers.",
        "product_categories": [
            "fire extinguisher",
            "wheeled fire extinguisher"
        ],
        "applications": [
            "industrial fire safety",
            "fire protection",
            "industrial facilities"
        ],
        "technical_requirements": [
            "fire extinguishing performance",
            "construction",
            "performance requirements"
        ],
        "keywords": [
            "wheeled fire extinguisher",
            "fire extinguisher",
            "industrial fire protection"
        ],
        "related_standards": [],
        "amendments": [],
        "status": "verified_record",
        "status_verified": True,
        "certification_applicable": True,
        "mandatory": True,
        "qco": "Fire Extinguishers Quality Control Order, 2023",
        "source_url":
            "https://www.bis.gov.in/product-certification/products-under-compulsory-certification/scheme-i-mark-scheme/?lang=en",
        "evidence":
            "BIS Scheme-I information lists IS 16018:2012 for Wheeled Fire Extinguishers under the Fire Extinguishers Quality Control Order, 2023.",
        "last_verified": "2026-08-29",
        "domain": "fire_safety",
        "verification_level": "A",
        "evidence_type": "official_bis_certification"
    }

])

print("Total new records:", len(new_records))

Total new records: 4


Add plumbing records

In [ ]:
new_records.extend([

    {
        "standard_id": "IS 2556",
        "part": "Part 1",
        "section": "",
        "edition": "2021",
        "title":
            "Vitreous China Sanitary Appliances — Specification Part 1 General Requirements",
        "scope":
            "General requirements for vitreous china sanitary appliances.",
        "product_categories": [
            "sanitary appliances",
            "vitreous china sanitary appliances"
        ],
        "applications": [
            "plumbing",
            "sanitary installations",
            "building services"
        ],
        "technical_requirements": [
            "material",
            "dimensions",
            "performance",
            "water efficiency"
        ],
        "keywords": [
            "sanitary",
            "plumbing",
            "wash basin",
            "vitreous china"
        ],
        "related_standards": [],
        "amendments": [],
        "status": "verified_record",
        "status_verified": True,
        "certification_applicable": None,
        "mandatory": None,
        "qco": None,
        "source_url":
            "https://www.bis.gov.in/wp-content/uploads/2022/01/ARTICLE-FOR-WHATS-NEW-NEW-STANDARDS-FOR-WATER-EFFICIENT-PLUMBING-PRODUCTS-NEW.pdf",
        "evidence":
            "Listed by BIS in its publication on water-efficient plumbing products.",
        "last_verified": "2026-08-29",
        "domain": "plumbing",
        "verification_level": "A",
        "evidence_type": "official_bis_publication"
    },

    {
        "standard_id": "IS 774",
        "part": "",
        "section": "",
        "edition": "2021",
        "title":
            "Ceramic Vitreous China Flushing Cisterns for Water Closets and Urinals — Specification",
        "scope":
            "Requirements for ceramic vitreous china flushing cisterns.",
        "product_categories": [
            "flushing cistern",
            "sanitary plumbing"
        ],
        "applications": [
            "plumbing",
            "water closets",
            "urinals",
            "building services"
        ],
        "technical_requirements": [
            "flushing performance",
            "water efficiency",
            "construction"
        ],
        "keywords": [
            "flushing cistern",
            "water closet",
            "urinal",
            "plumbing"
        ],
        "related_standards": [],
        "amendments": [],
        "status": "verified_record",
        "status_verified": True,
        "certification_applicable": None,
        "mandatory": None,
        "qco": None,
        "source_url":
            "https://www.bis.gov.in/wp-content/uploads/2022/01/ARTICLE-FOR-WHATS-NEW-NEW-STANDARDS-FOR-WATER-EFFICIENT-PLUMBING-PRODUCTS-NEW.pdf",
        "evidence":
            "Listed by BIS in its publication on water-efficient plumbing products.",
        "last_verified": "2026-08-29",
        "domain": "plumbing",
        "verification_level": "A",
        "evidence_type": "official_bis_publication"
    }
])

print("Total new records:", len(new_records))

Total new records: 6


Merge them into MASTER_KB

In [ ]:
MASTER_KB.extend(new_records)

MASTER_KB = remove_duplicate_standards(
    MASTER_KB
)

print(
    "MASTER KB now contains:",
    len(MASTER_KB),
    "records"
)

MASTER KB now contains: 10 records


Check the database

In [ ]:
for record in MASTER_KB:

    print(
        f"{record['standard_id']:12} | "
        f"{record.get('edition', ''):8} | "
        f"{record.get('domain', ''):15} | "
        f"{record.get('verification_level', '')}"
    )

IS 7098      | 2025     | electrical      | A
IS 10322     | 2026     | electrical      | A
IS 16107     | 2017     | electrical      | A
IS 269       | 2015     | construction    | A
IS 455       | 2015     | construction    | A
IS 1489      | 2015     | construction    | A
IS 15683     | 2018     | fire_safety     | A
IS 16018     | 2012     | fire_safety     | A
IS 2556      | 2021     | plumbing        | A
IS 774       | 2021     | plumbing        | A


Save the expanded KB

In [ ]:
with open(
    "knowledge_base/master_indian_standards.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MASTER_KB,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Expanded master KB saved.")

Expanded master KB saved.


Rebuild the embedding database

In [ ]:
embedding_documents = []

for record in MASTER_KB:

    embedding_documents.append({
        "standard_id": record["standard_id"],
        "text": build_embedding_text(record)
    })

texts = [
    x["text"]
    for x in embedding_documents
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embedding_matrix = np.asarray(
    embeddings,
    dtype="float32"
)

real_faiss_index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

real_faiss_index.add(
    embedding_matrix
)

print(
    "FAISS vectors:",
    real_faiss_index.ntotal
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS vectors: 10


Save everything

In [ ]:
faiss.write_index(
    real_faiss_index,
    "knowledge_base/real_indian_standards.faiss"
)

with open(
    "knowledge_base/embedding_documents.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        embedding_documents,
        f,
        indent=2,
        ensure_ascii=False
    )

print("FAISS + metadata saved.")

FAISS + metadata saved.


Test

In [ ]:
inspect_retrieval("""
We need cement for construction
of a residential building.
""")


RANK: 1
STANDARD: IS 269
TITLE: Ordinary Portland Cement — Specification (Sixth Revision)
SCORE: 0.483

PRODUCTS:
cement, ordinary portland cement, OPC

APPLICATIONS:
building construction, residential construction, civil construction, concrete construction

TECHNICAL REQUIREMENTS:
fineness, setting time, soundness, compressive strength, chemical requirements


IndexError: list index out of range

In [ ]:
# Synchronize metadata with the newly rebuilt FAISS index

real_standards = MASTER_KB

print("FAISS vectors :", real_faiss_index.ntotal)
print("KB records    :", len(real_standards))

FAISS vectors : 10
KB records    : 10


In [ ]:
for i, record in enumerate(real_standards):
    print(i, record["standard_id"], "|", record["title"])

0 IS 7098 | Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts
1 IS 10322 | Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting
2 IS 16107 | Luminaires Performance Part 2 Particular Requirements Section 2 LED Street Lighting Luminaire
3 IS 269 | Ordinary Portland Cement — Specification (Sixth Revision)
4 IS 455 | Portland Slag Cement — Specification
5 IS 1489 | Portland Pozzolana Cement — Specification: Part 1 Fly Ash Based
6 IS 15683 | Portable Fire Extinguishers
7 IS 16018 | Wheeled Fire Extinguishers
8 IS 2556 | Vitreous China Sanitary Appliances — Specification Part 1 General Requirements
9 IS 774 | Ceramic Vitreous China Flushing Cisterns for Water Closets and Urinals — Specification


In [ ]:
def inspect_retrieval(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # Never request more results than actually exist
    k = min(top_k, real_faiss_index.ntotal)

    scores, indices = real_faiss_index.search(
        query_embedding,
        k
    )

    print("\n" + "=" * 70)
    print("QUERY")
    print(query)
    print("=" * 70)

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        idx = int(idx)

        # Safety check
        if idx < 0 or idx >= len(real_standards):
            print(
                f"\nWARNING: FAISS returned index {idx}, "
                f"but KB contains only {len(real_standards)} records."
            )
            continue

        record = real_standards[idx]

        print("\n" + "=" * 70)

        print("RANK:", rank)
        print(
            "STANDARD:",
            record.get("standard_id", "")
        )

        print(
            "TITLE:",
            record.get("title", "")
        )

        print(
            "SCORE:",
            round(float(score), 3)
        )

        print("\nPRODUCTS:")

        products = record.get(
            "product_categories",
            []
        )

        print(
            ", ".join(products)
            if products
            else "Not available"
        )

        print("\nAPPLICATIONS:")

        applications = record.get(
            "applications",
            []
        )

        print(
            ", ".join(applications)
            if applications
            else "Not available"
        )

        print("\nTECHNICAL REQUIREMENTS:")

        requirements = record.get(
            "technical_requirements",
            []
        )

        print(
            ", ".join(requirements)
            if requirements
            else "Not available"
        )

    print("\n" + "=" * 70)

In [ ]:
inspect_retrieval("""
We need cement for construction
of a residential building.
""")


QUERY

We need cement for construction
of a residential building.


RANK: 1
STANDARD: IS 269
TITLE: Ordinary Portland Cement — Specification (Sixth Revision)
SCORE: 0.483

PRODUCTS:
cement, ordinary portland cement, OPC

APPLICATIONS:
building construction, residential construction, civil construction, concrete construction

TECHNICAL REQUIREMENTS:
fineness, setting time, soundness, compressive strength, chemical requirements

RANK: 2
STANDARD: IS 455
TITLE: Portland Slag Cement — Specification
SCORE: 0.389

PRODUCTS:
cement, portland slag cement

APPLICATIONS:
construction, building construction, concrete construction

TECHNICAL REQUIREMENTS:
chemical requirements, physical requirements, strength, setting time

RANK: 3
STANDARD: IS 1489
TITLE: Portland Pozzolana Cement — Specification: Part 1 Fly Ash Based
SCORE: 0.388

PRODUCTS:
cement, portland pozzolana cement, PPC

APPLICATIONS:
construction, building construction, concrete construction

TECHNICAL REQUIREMENTS:
chemical requireme

In [ ]:
inspect_retrieval("""
We need portable fire extinguishers
for an industrial manufacturing facility.
""")


QUERY

We need portable fire extinguishers
for an industrial manufacturing facility.


RANK: 1
STANDARD: IS 15683
TITLE: Portable Fire Extinguishers
SCORE: 0.583

PRODUCTS:
fire extinguisher, portable fire extinguisher

APPLICATIONS:
fire safety, industrial facilities, commercial buildings, building fire protection

TECHNICAL REQUIREMENTS:
fire extinguishing performance, construction, performance requirements

RANK: 2
STANDARD: IS 16018
TITLE: Wheeled Fire Extinguishers
SCORE: 0.486

PRODUCTS:
fire extinguisher, wheeled fire extinguisher

APPLICATIONS:
industrial fire safety, fire protection, industrial facilities

TECHNICAL REQUIREMENTS:
fire extinguishing performance, construction, performance requirements

RANK: 3
STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts
SCORE: 0.284

PRODUCTS:
electrical cables, power cables, XLPE cables, electrical wire, FR cable, FRLSH cab

In [ ]:
inspect_retrieval("""
We need water-efficient flushing
cisterns for a commercial building.
""")


QUERY

We need water-efficient flushing
cisterns for a commercial building.


RANK: 1
STANDARD: IS 774
TITLE: Ceramic Vitreous China Flushing Cisterns for Water Closets and Urinals — Specification
SCORE: 0.447

PRODUCTS:
flushing cistern, sanitary plumbing

APPLICATIONS:
plumbing, water closets, urinals, building services

TECHNICAL REQUIREMENTS:
flushing performance, water efficiency, construction

RANK: 2
STANDARD: IS 2556
TITLE: Vitreous China Sanitary Appliances — Specification Part 1 General Requirements
SCORE: 0.271

PRODUCTS:
sanitary appliances, vitreous china sanitary appliances

APPLICATIONS:
plumbing, sanitary installations, building services

TECHNICAL REQUIREMENTS:
material, dimensions, performance, water efficiency

RANK: 3
STANDARD: IS 10322
TITLE: Luminaires — Part 5: Particular Requirements Section 3: Luminaires for Road and Street Lighting
SCORE: 0.232

PRODUCTS:
road lighting luminaires, street lighting luminaires, LED street lights, outdoor lighting

APPLICATIONS:


In [ ]:
inspect_retrieval("""
We need electrical cable for an industrial
installation with fire-resistant requirements.
""")


QUERY

We need electrical cable for an industrial
installation with fire-resistant requirements.


RANK: 1
STANDARD: IS 7098
TITLE: Cross-linked Polyethylene Insulated Thermoplastic Sheathed Cables — Specification Part 1 for Working Voltages up to and Including 1100 Volts
SCORE: 0.481

PRODUCTS:
electrical cables, power cables, XLPE cables, electrical wire, FR cable, FRLSH cable, LSHF cable

APPLICATIONS:
electrical installations, power distribution, industrial electrical installations

TECHNICAL REQUIREMENTS:
conductor requirements, insulation requirements, sheath requirements, electrical performance, mechanical performance, fire performance, flame resistance, FR cable requirements, FRLSH cable requirements, LSHF cable requirements

RANK: 2
STANDARD: IS 15683
TITLE: Portable Fire Extinguishers
SCORE: 0.332

PRODUCTS:
fire extinguisher, portable fire extinguisher

APPLICATIONS:
fire safety, industrial facilities, commercial buildings, building fire protection

TECHNICAL REQUIREMENTS:


In [ ]:
!pwd
!ls -lah

/content
total 28K
drwxr-xr-x 1 root root 4.0K Aug 28 18:45 .
drwxr-xr-x 1 root root 4.0K Aug 28 18:34 ..
drwxr-xr-x 4 root root 4.0K Aug 24 13:27 .config
drwxr-xr-x 2 root root 4.0K Aug 28 18:43 indian_standards_rag
drwxr-xr-x 2 root root 4.0K Aug 28 19:01 knowledge_base
drwxr-xr-x 2 root root 4.0K Aug 28 18:43 real_indian_standards_rag
drwxr-xr-x 1 root root 4.0K Aug 24 13:28 sample_data


In [ ]:
!git --version

git version 2.34.1


In [ ]:
!git config --global user.name "SanskrutiMandavkar20"
!git config --global user.email "smandavkar2006@gmail.com"

In [ ]:
!find /content -maxdepth 2 -type d | head -30

/content
/content/.config
/content/.config/logs
/content/.config/configurations
/content/indian_standards_rag
/content/knowledge_base
/content/real_indian_standards_rag
/content/sample_data


In [ ]:
%cd /content/IndianStandardAI

[Errno 2] No such file or directory: '/content/IndianStandardAI'
/content


In [ ]:
%cd /content/real_indian_standards_rag

/content/real_indian_standards_rag


In [ ]:
!ls -lah

total 20K
drwxr-xr-x 2 root root 4.0K Aug 28 18:43 .
drwxr-xr-x 1 root root 4.0K Aug 28 18:45 ..
-rw-r--r-- 1 root root 4.6K Aug 28 18:43 standards.index
-rw-r--r-- 1 root root 3.3K Aug 28 18:43 standards_metadata.json


In [ ]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/real_indian_standards_rag/.git/


In [ ]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	standards.index
	standards_metadata.json

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!find /content -maxdepth 3 -type f \
  \( -name "*.ipynb" -o -name "*.py" -o -name "*.json" \) \
  | sort | head -200

/content/.config/.last_update_check.json
/content/indian_standards_rag/standards_metadata.json
/content/knowledge_base/embedding_documents.json
/content/knowledge_base/indian_standards.json
/content/knowledge_base/master_indian_standards.json
/content/real_indian_standards_rag/standards_metadata.json
/content/sample_data/anscombe.json


In [ ]:
!du -sh /content/* 2>/dev/null



16K	/content/indian_standards_rag
64K	/content/knowledge_base
136K	/content/real_indian_standards_rag
55M	/content/sample_data


In [ ]:
!find /content -maxdepth 3 -type f -name "*.ipynb" -print

In [ ]:
!find /content -maxdepth 3 -type f -name "*.py" -print

In [ ]:

%cd /content
!mkdir -p IndianStandardAI
!cp -r /content/knowledge_base /content/IndianStandardAI/
!cp -r /content/real_indian_standards_rag /content/IndianStandardAI/

/content


In [ ]:
!find /content/IndianStandardAI -maxdepth 3 -type f -print

/content/IndianStandardAI/knowledge_base/real_indian_standards.faiss
/content/IndianStandardAI/knowledge_base/indian_standards.csv
/content/IndianStandardAI/knowledge_base/indian_standards.json
/content/IndianStandardAI/knowledge_base/master_indian_standards.csv
/content/IndianStandardAI/knowledge_base/embedding_documents.json
/content/IndianStandardAI/knowledge_base/master_indian_standards.json
/content/IndianStandardAI/real_indian_standards_rag/standards.index
/content/IndianStandardAI/real_indian_standards_rag/.git/HEAD
/content/IndianStandardAI/real_indian_standards_rag/.git/config
/content/IndianStandardAI/real_indian_standards_rag/.git/description
/content/IndianStandardAI/real_indian_standards_rag/standards_metadata.json
/content/IndianStandardAI/real_indian_standards_rag/.gitignore


In [ ]:
%cd /content/IndianStandardAI

!rm -rf real_indian_standards_rag/.git

/content/IndianStandardAI


In [ ]:
!find . -maxdepth 3 -type f -print

./knowledge_base/real_indian_standards.faiss
./knowledge_base/indian_standards.csv
./knowledge_base/indian_standards.json
./knowledge_base/master_indian_standards.csv
./knowledge_base/embedding_documents.json
./knowledge_base/master_indian_standards.json
./real_indian_standards_rag/standards.index
./real_indian_standards_rag/standards_metadata.json
./real_indian_standards_rag/.gitignore


In [ ]:
%%writefile /content/IndianStandardAI/.gitignore
# Python
__pycache__/
*.py[cod]

# Jupyter
.ipynb_checkpoints/

# Secrets
.env
.env.*
*.key
credentials.json
token.json

# Virtual environments
.venv/
venv/
env/

# Datasets / large data
datasets/
*.parquet

# Generated vector indexes
*.faiss
*.index

# Model weights
*.bin
*.safetensors
*.pt
*.pth
*.gguf

# Temporary files
tmp/
temp/
logs/

Writing /content/IndianStandardAI/.gitignore


In [ ]:
%cd /content/IndianStandardAI
!git init
!git branch -M main
!git status --short

/content/IndianStandardAI
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/IndianStandardAI/.git/
?? .gitignore
?? knowledge_base/
?? real_indian_standards_rag/


In [1]:
%cd /content/IndianStandardAI
!git status --short

[Errno 2] No such file or directory: '/content/IndianStandardAI'
/content
fatal: not a git repository (or any of the parent directories): .git


In [2]:
!ls -lah /content

total 16K
drwxr-xr-x 1 root root 4.0K Aug 24 13:21 .
drwxr-xr-x 1 root root 4.0K Aug 29 07:29 ..
drwxr-xr-x 4 root root 4.0K Aug 24 13:21 .config
drwxr-xr-x 1 root root 4.0K Aug 24 13:21 sample_data


In [1]:
import sys
import numpy as np
import pandas as pd
import requests
import faiss
import sentence_transformers

print("✅ Environment working!")
print("Python:", sys.version)
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS:", faiss.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

✅ Environment working!
Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Python executable: d:\Indian-Standards-AI\.venv\Scripts\python.exe
NumPy: 2.5.2
Pandas: 3.0.5
FAISS: 1.15.0
Sentence Transformers: 6.0.0


In [1]:
import requests

url = "http://127.0.0.1:11434/api/generate"

payload = {
    "model": "qwen2.5:3b",
    "prompt": "Explain in one sentence what an Indian Standard is.",
    "stream": False
}

response = requests.post(url, json=payload, timeout=120)

print("HTTP status:", response.status_code)
print(response.text)

HTTP status: 200
{"model":"qwen2.5:3b","created_at":"2026-08-29T10:01:17.5208811Z","response":"An Indian Standard is a specification or set of guidelines established by the Indian Standards Institute for ensuring uniformity and quality in various products, materials, and services in India.","done":true,"done_reason":"stop","context":[151644,8948,198,2610,525,1207,16948,11,3465,553,54364,14817,13,1446,525,264,10950,17847,13,151645,198,151644,872,198,840,20772,304,825,11652,1128,458,7748,11766,374,13,151645,198,151644,77091,198,2082,7748,11766,374,264,25128,476,738,315,17501,9555,553,279,7748,34553,9976,369,22573,13794,487,323,4271,304,5257,3871,11,7236,11,323,3516,304,6747,13],"total_duration":4398042500,"load_duration":3720700,"prompt_eval_count":40,"prompt_eval_duration":415743000,"eval_count":34,"eval_duration":3963808000}
